In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Cell 2 — dataset discovery

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

def discover_dataset_root():
    # Look recursively for tasks.jsonl
    matches = list(INPUT.rglob("tasks.jsonl"))
    for task_file in matches:
        root = task_file.parent
        if (root / "sample_submission").exists():
            return root
    raise FileNotFoundError("Could not find dataset root containing tasks.jsonl and sample_submission")

DATASET_ROOT = discover_dataset_root()

TASKS_PATH = DATASET_ROOT / "tasks.jsonl"
HARNESS_README = DATASET_ROOT / "HARNESS_README.md"
SAMPLE_SUBMISSION = DATASET_ROOT / "sample_submission"
SNAPSHOTS_DIR = DATASET_ROOT / "snapshots"
GRAPHS_DIR = DATASET_ROOT / "graphs"
EMBEDDINGS_DIR = DATASET_ROOT / "embeddings"
WHEELS_DIR = DATASET_ROOT / "wheels"
SANDBOX_DIR = DATASET_ROOT / "sandbox"
DOCKER_DIR = DATASET_ROOT / "docker"

print("DATASET_ROOT:", DATASET_ROOT)
print("TASKS_PATH exists:", TASKS_PATH.exists(), TASKS_PATH)
print("HARNESS_README exists:", HARNESS_README.exists(), HARNESS_README)
print("SAMPLE_SUBMISSION exists:", SAMPLE_SUBMISSION.exists(), SAMPLE_SUBMISSION)
print("SNAPSHOTS_DIR exists:", SNAPSHOTS_DIR.exists(), SNAPSHOTS_DIR)
print("GRAPHS_DIR exists:", GRAPHS_DIR.exists(), GRAPHS_DIR)
print("EMBEDDINGS_DIR exists:", EMBEDDINGS_DIR.exists(), EMBEDDINGS_DIR)
print("WHEELS_DIR exists:", WHEELS_DIR.exists(), WHEELS_DIR)
print("SANDBOX_DIR exists:", SANDBOX_DIR.exists(), SANDBOX_DIR)
print("DOCKER_DIR exists:", DOCKER_DIR.exists(), DOCKER_DIR)

Cell 3 — quick inventory

In [ ]:
def preview_tree(root, limit=60):
    print(f"\nListing from: {root}")
    count = 0
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        print(rel)
        count += 1
        if count >= limit:
            print("... truncated ...")
            break

preview_tree(DATASET_ROOT, limit=80)
preview_tree(SAMPLE_SUBMISSION, limit=80)

Cell 4 — load tasks.jsonl

In [ ]:
df = pd.read_json(TASKS_PATH, lines=True, dtype=False)

print("Tasks:", len(df))
print("Columns:", list(df.columns))
print("Repos:", df["repo"].nunique() if "repo" in df.columns else "n/a")

df.head(3)

Cell 5 — patch parser / localization labels

In [ ]:
import re
from pathlib import Path

HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@(.*)$")
SYMBOL_RE = re.compile(r"(?:async\s+)?(?:def|class)\s+([A-Za-z_]\w*)")

def _clean_path(p: str) -> str:
    p = p.split("\t")[0].strip()
    return p[2:] if p.startswith(("a/", "b/")) else p

def parse_patch(text: str):
    files, cur, old_path = [], None, None
    rem_old = rem_new = 0

    def new_file(path):
        f = {
            "path": path,
            "added": 0,
            "deleted": 0,
            "hunks": 0,
            "symbols": set(),
            "_hdr": False,
        }
        files.append(f)
        return f

    for line in (text or "").splitlines():
        if rem_old > 0 or rem_new > 0:
            tag = line[:1]
            if tag == "+":
                cur["added"] += 1
                rem_new -= 1
                m = SYMBOL_RE.match(line[1:].strip())
                if m:
                    cur["symbols"].add(m.group(1))
                continue
            if tag == "-":
                cur["deleted"] += 1
                rem_old -= 1
                m = SYMBOL_RE.match(line[1:].strip())
                if m:
                    cur["symbols"].add(m.group(1))
                continue
            if tag == " " or line == "":
                rem_old -= 1
                rem_new -= 1
                continue
            if tag == "\\":
                continue
            rem_old = rem_new = 0

        if line.startswith("diff --git "):
            m = re.match(r"diff --git a/(\S+) b/(\S+)", line)
            cur = new_file(m.group(2) if m else line[11:].strip())
            continue

        if line.startswith("--- "):
            old_path = _clean_path(line[4:])
            continue

        if line.startswith("+++ "):
            new_path = _clean_path(line[4:])
            if cur is None or cur["_hdr"]:
                cur = new_file(new_path)
            cur["_hdr"] = True
            cur["path"] = new_path if new_path != "/dev/null" else old_path
            continue

        m = HUNK_RE.match(line)
        if m and cur is not None:
            rem_old = int(m.group(2)) if m.group(2) is not None else 1
            rem_new = int(m.group(4)) if m.group(4) is not None else 1
            cur["hunks"] += 1
            sm = SYMBOL_RE.search(m.group(5) or "")
            if sm:
                cur["symbols"].add(sm.group(1))

    for f in files:
        f["symbols"] = sorted(f["symbols"])
        f.pop("_hdr", None)

    return files

def is_test_path(path: str) -> bool:
    p = path.lower()
    return (
        "/tests/" in p or
        "/test_" in p or
        p.endswith("_test.py") or
        p.endswith("conftest.py")
    )

def patch_features(row):
    files = parse_patch(row.get("patch", ""))
    src_files = [f for f in files if not is_test_path(f["path"])]
    return pd.Series({
        "gold_files": [f["path"] for f in src_files],
        "gold_symbols": sorted({s for f in src_files for s in f["symbols"]}),
        "n_files_changed": len(src_files),
        "n_hunks": sum(f["hunks"] for f in src_files),
        "lines_added": sum(f["added"] for f in src_files),
        "lines_deleted": sum(f["deleted"] for f in src_files),
    })

feat = df.apply(patch_features, axis=1)
df = pd.concat([df, feat], axis=1)
df["patch_churn"] = df["lines_added"] + df["lines_deleted"]
df["issue_words"] = df["problem_statement"].fillna("").str.split().str.len()

df[[
    "instance_id", "repo", "gold_files", "gold_symbols",
    "n_files_changed", "n_hunks", "patch_churn"
]].head(10)

Cell 6 — EDA

In [ ]:
print("Median patch churn:", int(df["patch_churn"].median()))
print("Single-file rate:", round((df["n_files_changed"] == 1).mean(), 3))
print("Multi-hunk rate:", round((df["n_hunks"] > 1).mean(), 3))

hot_files = (
    df[["repo", "gold_files"]]
    .explode("gold_files")
    .dropna()
    .groupby(["repo", "gold_files"])
    .size()
    .sort_values(ascending=False)
    .reset_index(name="tasks")
)

hot_files.head(20)

Cell 7 — read harness

In [ ]:
if HARNESS_README.exists():
    harness_text = HARNESS_README.read_text(encoding="utf-8", errors="ignore")
    print(harness_text[:6000])
else:
    print("HARNESS_README.md not found")

Cell 8 — tool inventory

In [ ]:
KNOWN_TOOLS = [
    "run_command",
    "submit_patch",
    "get_status",
    "read_file",
    "edit_file",
    "write_file",
    "get_code_neighbors",
    "search_similar_code",
    "get_code_subgraph",
]

tool_inventory = {tool: (tool in harness_text) for tool in KNOWN_TOOLS}
tool_inventory

Cell 9 — baseline prompt text

In [ ]:
MAX_SEARCH_STEPS = 6
MAX_FILE_READS = 6

BASELINE_SYSTEM_PROMPT = f"""
You are an offline software-engineering agent operating inside the Gemma 4 Developer Agent competition harness.

Your goal is to resolve the issue by producing the smallest correct source patch.

High-level policy:
1. Restate the bug in 1-3 sentences.
2. Localize before editing. Prefer semantic/code-graph evidence over broad wandering.
3. Read only the most relevant files and symbols.
4. Form a concrete defect hypothesis before making edits.
5. Prefer minimal, behavior-preserving fixes.
6. Avoid refactors, cleanup, or stylistic rewrites unless required for correctness.
7. Verify narrowly when possible using focused commands or reproductions.
8. If the location is still too uncertain after bounded search, avoid a low-confidence patch.

Search budget guidance:
- At most {MAX_SEARCH_STEPS} meaningful search steps before deciding.
- At most {MAX_FILE_READS} targeted file reads before switching from search to diagnosis.

Editing guidance:
- Preserve existing interfaces unless the issue clearly requires a change.
- Reuse existing helper patterns from nearby code.
- Do not modify tests unless the task truly requires it.

Verification guidance:
- If a narrow reproduction or focused test exists, use it first.
- Prefer high-signal checks over broad expensive runs.
- Remove temporary debugging artifacts before submitting.

Final output behavior:
- Only submit a patch once you believe it is the best bounded attempt.
- If no credible fix is found, prefer no patch over random edits.
""".strip()

print(BASELINE_SYSTEM_PROMPT[:2000])

Cell 10 — copy sample_submission/ into working dir

In [ ]:
# import shutil
# import zipfile

# SUBMISSION_DIR = WORKING / "submission"
# SUBMISSION_ZIP = WORKING / "submission.zip"

# if SUBMISSION_DIR.exists():
#     shutil.rmtree(SUBMISSION_DIR)

# shutil.copytree(SAMPLE_SUBMISSION, SUBMISSION_DIR)

# print("Copied sample_submission to:", SUBMISSION_DIR)

Cell 11 — inspect copied config files

In [ ]:
# config_files = [p for p in SUBMISSION_DIR.rglob("*") if p.suffix.lower() in {".yaml", ".yml", ".md", ".txt"}]

# for p in config_files[:20]:
#     print("=" * 100)
#     print(p.relative_to(SUBMISSION_DIR))
#     try:
#         print(p.read_text(encoding="utf-8", errors="ignore")[:1500])
#     except Exception as e:
#         print("Unreadable:", e)

Cell 12 — replace prompt file

In [ ]:
# prompt_candidates = [
#     p for p in SUBMISSION_DIR.rglob("*")
#     if p.is_file() and p.suffix.lower() in {".md", ".txt"} and "prompt" in str(p).lower()
# ]

# if not prompt_candidates:
#     fallback = SUBMISSION_DIR / "prompts" / "system.md"
#     fallback.parent.mkdir(parents=True, exist_ok=True)
#     fallback.write_text(BASELINE_SYSTEM_PROMPT, encoding="utf-8")
#     target_prompt = fallback
# else:
#     ranked = sorted(
#         prompt_candidates,
#         key=lambda p: (0 if p.name.lower() == "system.md" else 1, len(str(p)))
#     )
#     target_prompt = ranked[0]
#     target_prompt.write_text(BASELINE_SYSTEM_PROMPT, encoding="utf-8")

# print("Updated prompt file:", target_prompt)

Cell 13 — build submission.zip

In [ ]:
# if SUBMISSION_ZIP.exists():
#     SUBMISSION_ZIP.unlink()

# with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
#     for path in sorted(SUBMISSION_DIR.rglob("*")):
#         if path.is_file():
#             zf.write(path, path.relative_to(SUBMISSION_DIR))

# print("Created:", SUBMISSION_ZIP)
# print("Size (MB):", round(SUBMISSION_ZIP.stat().st_size / (1024 * 1024), 3))

Cell 14 — Sanity check the final archive

In [ ]:
# with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
#     names = zf.namelist()
#     print("Archive entries:", len(names))
#     print("First 50 entries:")
#     for name in names[:50]:
#         print("-", name)

# assert "agent.yaml" in names, "Expected agent.yaml at archive root"
# print("Sanity check passed: agent.yaml is present at zip root.")

Cell 15 — Graph loader + index builder

In [ ]:
# Cell 15 — Robust graph loader / index builder (empty-file tolerant)
import json
import gzip
import re
from collections import defaultdict
from pathlib import Path

GRAPH_CACHE = {}
GRAPH_INDEX_CACHE = {}
BAD_GRAPH_PATHS = set()

STOPWORDS = {
    "the", "and", "for", "with", "from", "that", "this", "when", "where", "into",
    "should", "would", "could", "while", "does", "doesnt", "dont", "after", "before",
    "have", "has", "had", "been", "being", "then", "than", "them", "they", "their",
    "there", "about", "because", "issue", "error", "errors", "bug", "fails", "failure",
    "request", "response", "return", "returns", "support", "handle", "handling"
}

TOKEN_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]+")

def repo_short(repo_name: str) -> str:
    return repo_name.split("/")[-1]

def normalize_tokens(text: str):
    toks = [t.lower() for t in TOKEN_RE.findall(text or "")]
    toks = [t for t in toks if len(t) >= 3 and t not in STOPWORDS]
    return toks

def is_usable_graph_file(path: Path) -> bool:
    return path is not None and path.exists() and path.is_file() and path.stat().st_size > 0

def graph_candidates_for_row(row):
    repo_name = repo_short(row["repo"])
    base_commit = str(row["base_commit"])

    candidates = [
        GRAPHS_DIR / f"{repo_name}_{base_commit}.json",   # prefer commit graph
        GRAPHS_DIR / f"{row['instance_id']}.json",        # fallback task graph
    ]

    # also allow shortened commit fallback
    short = base_commit[:12]
    candidates.extend(sorted(GRAPHS_DIR.glob(f"{repo_name}_{short}*.json")))

    # remove duplicates while preserving order
    seen = set()
    uniq = []
    for p in candidates:
        if p not in seen:
            seen.add(p)
            uniq.append(p)
    return uniq

def graph_path_for_row(row):
    for p in graph_candidates_for_row(row):
        if is_usable_graph_file(p):
            return p
    return None

def inspect_file_header(path, n=120):
    path = Path(path)
    print("Path:", path)
    print("Exists:", path.exists())
    print("Size:", path.stat().st_size if path.exists() else "n/a", "bytes")
    if path.exists():
        raw = path.read_bytes()[:n]
        print("First bytes:", raw[:32])
        print("As text:", raw.decode("utf-8", errors="replace"))

def load_graph_json(path: Path):
    path = Path(path)

    if path in GRAPH_CACHE:
        return GRAPH_CACHE[path]

    raw = path.read_bytes()
    if len(raw) == 0:
        raise ValueError(f"Empty graph file: {path}")

    # gzip magic
    if raw[:2] == b"\x1f\x8b":
        raw = gzip.decompress(raw)

    # strip BOM
    if raw.startswith(b"\xef\xbb\xbf"):
        raw = raw[3:]

    text = raw.decode("utf-8", errors="replace").strip()
    if not text:
        raise ValueError(f"Decoded graph file is empty after stripping: {path}")

    if text.startswith("version https://git-lfs.github.com/spec/v1"):
        raise ValueError(f"Graph file looks like a Git LFS pointer: {path}")

    if text[0] in "{[":
        obj = json.loads(text)
        GRAPH_CACHE[path] = obj
        return obj

    # fallback: maybe first non-empty line is JSON
    lines = [line for line in text.splitlines() if line.strip()]
    if lines and lines[0][0] in "{[":
        obj = json.loads(lines[0])
        GRAPH_CACHE[path] = obj
        return obj

    raise ValueError(f"Unrecognized graph format: {path}\n{text[:200]}")

def build_graph_index(graph_obj: dict):
    nodes = graph_obj.get("nodes", [])
    edges = graph_obj.get("edges", graph_obj.get("links", []))

    node_rows = []
    node_by_id = {}
    adj = defaultdict(set)

    for node in nodes:
        nid = node.get("id") or node.get("name")
        if not nid:
            continue

        name = node.get("name", nid)
        text = node.get("text", "") or ""
        tail = str(nid).split(".")[-1]
        blob = f"{nid}\n{name}\n{text[:1200]}".lower()

        row = {
            "id": str(nid),
            "name": str(name),
            "tail": tail,
            "blob": blob,
            "preview": text[:220].replace("\n", " "),
        }
        node_rows.append(row)
        node_by_id[row["id"]] = row

    for edge in edges:
        src = edge.get("source")
        dst = edge.get("target")
        if src is None or dst is None:
            continue
        src, dst = str(src), str(dst)
        adj[src].add(dst)
        adj[dst].add(src)  # undirected expansion baseline

    return {
        "nodes": node_rows,
        "node_by_id": node_by_id,
        "adj": adj,
        "num_nodes": len(node_rows),
        "num_edges": len(edges),
    }

def get_graph_index(row):
    gpath = graph_path_for_row(row)
    if gpath is None:
        return None, None

    if gpath in BAD_GRAPH_PATHS:
        return None, None

    try:
        if gpath not in GRAPH_INDEX_CACHE:
            graph_obj = load_graph_json(gpath)
            GRAPH_INDEX_CACHE[gpath] = build_graph_index(graph_obj)
        return GRAPH_INDEX_CACHE[gpath], gpath
    except Exception as e:
        print(f"[graph-skip] {row['instance_id']} -> {gpath.name}: {e}")
        BAD_GRAPH_PATHS.add(gpath)
        return None, None

# quick sanity check
first_row = df.iloc[0]
graph_index, graph_path = get_graph_index(first_row)
print("Graph path:", graph_path)
if graph_index:
    print("Nodes:", graph_index["num_nodes"], "Edges:", graph_index["num_edges"])
    print("Sample node IDs:")
    for item in graph_index["nodes"][:5]:
        print("-", item["id"])
else:
    print("No usable graph found for first row")

Cell 16 — Direct symbol localization from issue text

In [ ]:
# Cell 16 — Direct symbol localization from issue text (safe)

def issue_query_tokens(row, max_tokens=24):
    text = (row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")
    toks = normalize_tokens(text)
    seen = set()
    out = []
    for t in toks:
        if t not in seen:
            seen.add(t)
            out.append(t)
        if len(out) >= max_tokens:
            break
    return out

def lexical_node_score(node, q_tokens, query_lc):
    blob = node["blob"]
    name_lc = node["name"].lower()
    tail_lc = node["tail"].lower()

    overlap = sum(tok in blob for tok in q_tokens)
    if overlap == 0:
        return 0.0

    score = float(overlap)

    if tail_lc in query_lc:
        score += 3.0
    if any(tok == tail_lc for tok in q_tokens):
        score += 3.0
    if any(tok in name_lc for tok in q_tokens):
        score += 1.0

    return score

def direct_symbol_candidates(row, top_k=15):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    query_text = ((row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")).lower()
    q_tokens = issue_query_tokens(row)

    scored = []
    for node in graph_index["nodes"]:
        s = lexical_node_score(node, q_tokens, query_text)
        if s > 0:
            scored.append({
                "id": node["id"],
                "tail": node["tail"],
                "name": node["name"],
                "score": round(s, 3),
                "preview": node["preview"],
            })

    scored = sorted(scored, key=lambda x: (-x["score"], x["id"]))
    return scored[:top_k]

def ranked_hits_gold_symbol(ranked_ids, gold_symbols):
    gold = set(gold_symbols or [])
    if not gold:
        return False
    for nid in ranked_ids:
        tail = nid.split(".")[-1]
        if tail in gold or nid in gold:
            return True
    return False

GRAPH_EVAL_N = 30
eligible = df[df["gold_symbols"].map(len) > 0].copy()
subset = eligible.sample(min(GRAPH_EVAL_N, len(eligible)), random_state=42)

direct_eval_rows = []
for _, row in subset.iterrows():
    ranked = direct_symbol_candidates(row, top_k=15)
    ranked_ids = [x["id"] for x in ranked]

    direct_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "gold_symbols": row["gold_symbols"],
        "direct_hit@1": ranked_hits_gold_symbol(ranked_ids[:1], row["gold_symbols"]),
        "direct_hit@5": ranked_hits_gold_symbol(ranked_ids[:5], row["gold_symbols"]),
        "direct_hit@10": ranked_hits_gold_symbol(ranked_ids[:10], row["gold_symbols"]),
        "num_ranked": len(ranked_ids),
        "graph_available": len(ranked_ids) > 0,
    })

direct_eval = pd.DataFrame(direct_eval_rows)

print("Direct symbol localization baseline")
print("Graph available rate:", round(direct_eval["graph_available"].mean(), 3))
print(direct_eval[["direct_hit@1", "direct_hit@5", "direct_hit@10"]].mean().round(3))

direct_eval.head(10)

Cell 17 — Graph-neighbor expansion baseline + example inspection

In [ ]:
# Cell 17 — Neighbor expansion over top symbols

def expanded_symbol_candidates(row, seed_top_k=8, final_top_k=25, neighbor_bonus=0.35):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    query_text = ((row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")).lower()
    q_tokens = issue_query_tokens(row)

    direct = direct_symbol_candidates(row, top_k=seed_top_k)
    if not direct:
        return []

    combined = {}
    for item in direct:
        combined[item["id"]] = float(item["score"])

    # expand through undirected graph neighborhood
    for item in direct:
        nid = item["id"]
        seed_score = float(item["score"])
        for nb in graph_index["adj"].get(nid, []):
            nb_node = graph_index["node_by_id"].get(nb)
            if nb_node is None:
                continue

            local_score = lexical_node_score(nb_node, q_tokens, query_text)
            total_score = local_score + neighbor_bonus * seed_score
            combined[nb] = max(combined.get(nb, 0.0), total_score)

    ranked = []
    for nid, score in combined.items():
        node = graph_index["node_by_id"][nid]
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(score, 3),
            "preview": node["preview"],
        })

    ranked = sorted(ranked, key=lambda x: (-x["score"], x["id"]))
    return ranked[:final_top_k]

neighbor_eval_rows = []
for _, row in subset.iterrows():
    direct_ranked = direct_symbol_candidates(row, top_k=10)
    expanded_ranked = expanded_symbol_candidates(row, seed_top_k=8, final_top_k=25)

    direct_ids = [x["id"] for x in direct_ranked]
    expanded_ids = [x["id"] for x in expanded_ranked]

    neighbor_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "direct_hit@10": ranked_hits_gold_symbol(direct_ids[:10], row["gold_symbols"]),
        "expanded_hit@10": ranked_hits_gold_symbol(expanded_ids[:10], row["gold_symbols"]),
        "expanded_hit@25": ranked_hits_gold_symbol(expanded_ids[:25], row["gold_symbols"]),
    })

neighbor_eval = pd.DataFrame(neighbor_eval_rows)

print("Direct vs graph-expanded localization")
print(pd.DataFrame({
    "direct_hit@10": [neighbor_eval["direct_hit@10"].mean()],
    "expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
}).round(3))

# Inspect one example task
example_row = subset.iloc[0]
print("\nExample task:", example_row["instance_id"], "| repo:", example_row["repo"])
print("Gold symbols:", example_row["gold_symbols"])
print("\nIssue snippet:")
print((example_row["problem_statement"] or "")[:800])

example_candidates = expanded_symbol_candidates(example_row, seed_top_k=8, final_top_k=12)
pd.DataFrame(example_candidates)[["id", "tail", "score", "preview"]]

Cell 18 — clean summary dashboard

In [ ]:
print("=== PUBLIC TASK SUMMARY ===")
print("Tasks:", len(df))
print("Repos:", df["repo"].nunique())
print("Median patch churn:", int(df["patch_churn"].median()))
print("Single-file rate:", round((df["n_files_changed"] == 1).mean(), 3))
print("Multi-hunk rate:", round((df["n_hunks"] > 1).mean(), 3))

print("\n=== GRAPH LOCALIZATION SUMMARY ===")
print("Graph available rate:", round(direct_eval["graph_available"].mean(), 3))
print("Direct hit@1 :", round(direct_eval["direct_hit@1"].mean(), 3))
print("Direct hit@5 :", round(direct_eval["direct_hit@5"].mean(), 3))
print("Direct hit@10:", round(direct_eval["direct_hit@10"].mean(), 3))

print("\nExpanded hit@10:", round(neighbor_eval["expanded_hit@10"].mean(), 3))
print("Expanded hit@25:", round(neighbor_eval["expanded_hit@25"].mean(), 3))

print("\n=== HOT FILES ===")
display(hot_files.head(15))

Cell 19 — inspect failed cases

In [ ]:
from IPython.display import display

failed_ids = neighbor_eval.loc[~neighbor_eval["expanded_hit@25"], "instance_id"].tolist()

print("Failed cases at expanded_hit@25:", len(failed_ids))
print("Showing first 5 failures...\n")

for iid in failed_ids[:5]:
    row = subset[subset["instance_id"] == iid].iloc[0]
    cands = expanded_symbol_candidates(row, seed_top_k=8, final_top_k=12)

    print("=" * 120)
    print("INSTANCE:", row["instance_id"])
    print("REPO:", row["repo"])
    print("GOLD SYMBOLS:", row["gold_symbols"])
    print("\nISSUE SNIPPET:")
    print((row["problem_statement"] or "")[:700])

    if len(cands) == 0:
        print("\nNo candidates returned.\n")
        continue

    print("\nTOP CANDIDATES:")
    display(pd.DataFrame(cands)[["id", "tail", "score", "preview"]])

Cell 20 — source-first reranking

In [ ]:
def repo_prefix_for_row(row):
    mapping = {
        "fastapi/fastapi": "fastapi.",
        "Textualize/rich": "rich.",
        "psf/requests": "requests.",
        "encode/httpx": "httpx.",
    }
    return mapping.get(row["repo"], "")

def is_test_node_id(nid: str) -> bool:
    nid = nid.lower()
    return (
        nid.startswith("tests.")
        or ".tests." in nid
        or ".test_" in nid
        or "conftest" in nid
    )

def source_first_rerank(row, candidates):
    prefix = repo_prefix_for_row(row)
    rescored = []

    for item in candidates:
        score2 = float(item["score"])
        nid = item["id"]

        if prefix and nid.startswith(prefix):
            score2 += 0.75

        if is_test_node_id(nid):
            score2 -= 2.5

        rescored.append({
            **item,
            "score2": round(score2, 3),
        })

    rescored = sorted(rescored, key=lambda x: (-x["score2"], x["id"]))
    return rescored

def expanded_symbol_candidates_source_first(row, seed_top_k=8, final_top_k=25):
    base = expanded_symbol_candidates(row, seed_top_k=seed_top_k, final_top_k=final_top_k)
    if not base:
        return []
    return source_first_rerank(row, base)[:final_top_k]

source_eval_rows = []
for _, row in subset.iterrows():
    ranked = expanded_symbol_candidates_source_first(row, seed_top_k=8, final_top_k=25)
    ranked_ids = [x["id"] for x in ranked]

    source_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "source_first_hit@10": ranked_hits_gold_symbol(ranked_ids[:10], row["gold_symbols"]),
        "source_first_hit@25": ranked_hits_gold_symbol(ranked_ids[:25], row["gold_symbols"]),
    })

source_eval = pd.DataFrame(source_eval_rows)

print("=== SOURCE-FIRST RERANK RESULTS ===")
print(pd.DataFrame({
    "expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
    "source_first_hit@10": [source_eval["source_first_hit@10"].mean()],
    "source_first_hit@25": [source_eval["source_first_hit@25"].mean()],
}).round(3))

Cell 21 — TF-IDF retriever over graph nodes

In [ ]:
# Cell 21 — TF-IDF retriever over graph node blobs
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

TFIDF_CACHE = {}

def get_tfidf_index(row):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return None, None, None

    if gpath in TFIDF_CACHE:
        return TFIDF_CACHE[gpath]

    texts = [node["blob"] for node in graph_index["nodes"]]
    if len(texts) == 0:
        TFIDF_CACHE[gpath] = (None, None, graph_index)
        return TFIDF_CACHE[gpath]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    X = vectorizer.fit_transform(texts)
    TFIDF_CACHE[gpath] = (vectorizer, X, graph_index)
    return TFIDF_CACHE[gpath]

def tfidf_symbol_candidates(row, top_k=15, return_score_map=False):
    vectorizer, X, graph_index = get_tfidf_index(row)
    if vectorizer is None or X is None or graph_index is None:
        return ({}, []) if return_score_map else []

    query_text = (row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")
    q = vectorizer.transform([query_text])

    if q.nnz == 0:
        return ({}, []) if return_score_map else []

    scores = (X @ q.T).toarray().ravel()
    order = np.argsort(-scores)

    ranked = []
    score_map = {}
    for idx in order:
        s = float(scores[idx])
        if s <= 0:
            break
        node = graph_index["nodes"][idx]
        score_map[node["id"]] = s
        ranked.append({
            "id": node["id"],
            "tail": node["tail"],
            "name": node["name"],
            "score": round(s, 4),
            "preview": node["preview"],
        })
        if len(ranked) >= top_k:
            break

    return (score_map, ranked) if return_score_map else ranked

# quick smoke test
smoke_row = subset.iloc[0]
tfidf_test = tfidf_symbol_candidates(smoke_row, top_k=10)
pd.DataFrame(tfidf_test)[["id", "tail", "score"]]

Cell 22 — TF-IDF + graph hybrid expansion

In [ ]:
# Cell 22 — TF-IDF seed retrieval + graph expansion

def hybrid_tfidf_graph_candidates(row, seed_top_k=8, final_top_k=25, neighbor_bonus=0.35):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    score_map, seeds = tfidf_symbol_candidates(row, top_k=seed_top_k, return_score_map=True)
    if not seeds:
        return []

    combined = {}

    # keep seed scores
    for item in seeds:
        combined[item["id"]] = float(item["score"])

    # expand neighbors
    for item in seeds:
        nid = item["id"]
        seed_score = float(item["score"])
        for nb in graph_index["adj"].get(nid, []):
            nb_base = float(score_map.get(nb, 0.0))
            total = nb_base + neighbor_bonus * seed_score
            combined[nb] = max(combined.get(nb, 0.0), total)

    ranked = []
    for nid, score in combined.items():
        node = graph_index["node_by_id"].get(nid)
        if node is None:
            continue
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(score, 4),
            "preview": node["preview"],
        })

    ranked = sorted(ranked, key=lambda x: (-x["score"], x["id"]))
    return ranked[:final_top_k]

hybrid_eval_rows = []
for _, row in subset.iterrows():
    tfidf_ranked = tfidf_symbol_candidates(row, top_k=10)
    hybrid_ranked = hybrid_tfidf_graph_candidates(row, seed_top_k=8, final_top_k=25)

    tfidf_ids = [x["id"] for x in tfidf_ranked]
    hybrid_ids = [x["id"] for x in hybrid_ranked]

    hybrid_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "tfidf_hit@10": ranked_hits_gold_symbol(tfidf_ids[:10], row["gold_symbols"]),
        "hybrid_hit@10": ranked_hits_gold_symbol(hybrid_ids[:10], row["gold_symbols"]),
        "hybrid_hit@25": ranked_hits_gold_symbol(hybrid_ids[:25], row["gold_symbols"]),
        "tfidf_candidates": len(tfidf_ids),
        "hybrid_candidates": len(hybrid_ids),
    })

hybrid_eval = pd.DataFrame(hybrid_eval_rows)

print("=== TF-IDF + GRAPH HYBRID RESULTS ===")
print(pd.DataFrame({
    "graph_direct_hit@10": [direct_eval["direct_hit@10"].mean()],
    "graph_expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "graph_expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
    "tfidf_hit@10": [hybrid_eval["tfidf_hit@10"].mean()],
    "hybrid_hit@10": [hybrid_eval["hybrid_hit@10"].mean()],
    "hybrid_hit@25": [hybrid_eval["hybrid_hit@25"].mean()],
}).round(3))

Cell 23 — inspect hybrid failures / wins

In [ ]:
# Cell 23 — Inspect where hybrid helps or still fails

compare = neighbor_eval.merge(
    hybrid_eval[["instance_id", "tfidf_hit@10", "hybrid_hit@10", "hybrid_hit@25"]],
    on="instance_id",
    how="left"
)

print("Cases where hybrid improved over graph-expanded@10:")
improved = compare[(compare["expanded_hit@10"] == False) & (compare["hybrid_hit@10"] == True)]
print(len(improved))
display(improved.head(10))

print("\nCases still failing at hybrid_hit@25:")
still_fail = compare[compare["hybrid_hit@25"] == False]
print(len(still_fail))
display(still_fail.head(10))

# show one improved example and one failed example
for label, frame in [("IMPROVED", improved), ("STILL_FAIL", still_fail)]:
    if len(frame) == 0:
        continue

    iid = frame.iloc[0]["instance_id"]
    row = subset[subset["instance_id"] == iid].iloc[0]
    cands = hybrid_tfidf_graph_candidates(row, seed_top_k=8, final_top_k=12)

    print("\n" + "=" * 120)
    print(label, "| INSTANCE:", row["instance_id"], "| REPO:", row["repo"])
    print("GOLD SYMBOLS:", row["gold_symbols"])
    print("\nISSUE SNIPPET:")
    print((row["problem_statement"] or "")[:700])

    if len(cands):
        display(pd.DataFrame(cands)[["id", "tail", "score", "preview"]])

Cell 24 — Clean query + source-only TF-IDF index

In [ ]:
# Cell 24 — Clean query + source-first TF-IDF
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

TFIDF2_CACHE = {}

def clean_query_text(row):
    text = (row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")
    text = re.sub(r"https?://\S+", " ", text)      # remove URLs
    text = text.replace("`", " ")
    text = text.replace("**", " ")
    text = re.sub(r"#+", " ", text)                # markdown headings
    text = re.sub(r"\s+", " ", text).strip()
    return text

def repo_prefix_for_row(row):
    mapping = {
        "fastapi/fastapi": "fastapi.",
        "Textualize/rich": "rich.",
        "psf/requests": "requests.",
        "encode/httpx": "httpx.",
    }
    return mapping.get(row["repo"], "")

def is_test_node_id(nid: str) -> bool:
    nid = nid.lower()
    return (
        nid.startswith("tests.")
        or ".tests." in nid
        or ".test_" in nid
        or "conftest" in nid
    )

def is_preferred_source_node(row, nid: str) -> bool:
    prefix = repo_prefix_for_row(row)
    if is_test_node_id(nid):
        return False
    if prefix and nid.startswith(prefix):
        return True
    return not is_test_node_id(nid)

def get_tfidf_index_source_first(row, source_only=True):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return None, None, None, None

    cache_key = (gpath, "source_only" if source_only else "all_nodes")
    if cache_key in TFIDF2_CACHE:
        return TFIDF2_CACHE[cache_key]

    selected_nodes = []
    for node in graph_index["nodes"]:
        nid = node["id"]
        if source_only:
            if not is_preferred_source_node(row, nid):
                continue
        selected_nodes.append(node)

    texts = [node["blob"] for node in selected_nodes]
    if len(texts) == 0:
        TFIDF2_CACHE[cache_key] = (None, None, None, graph_index)
        return TFIDF2_CACHE[cache_key]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    X = vectorizer.fit_transform(texts)

    TFIDF2_CACHE[cache_key] = (vectorizer, X, selected_nodes, graph_index)
    return TFIDF2_CACHE[cache_key]

def tfidf_symbol_candidates_source_first(row, top_k=15, source_only=True, return_score_map=False):
    vectorizer, X, selected_nodes, graph_index = get_tfidf_index_source_first(row, source_only=source_only)
    if vectorizer is None or X is None or selected_nodes is None:
        return ({}, []) if return_score_map else []

    query_text = clean_query_text(row)
    q = vectorizer.transform([query_text])

    if q.nnz == 0:
        return ({}, []) if return_score_map else []

    scores = (X @ q.T).toarray().ravel()
    order = np.argsort(-scores)

    ranked = []
    score_map = {}
    for idx in order:
        s = float(scores[idx])
        if s <= 0:
            break
        node = selected_nodes[idx]
        nid = node["id"]
        score_map[nid] = s
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(s, 4),
            "preview": node["preview"],
        })
        if len(ranked) >= top_k:
            break

    return (score_map, ranked) if return_score_map else ranked

# smoke test on prior bad case
row = subset[subset["instance_id"] == "rich_4077"].iloc[0]
clean_tfidf_test = tfidf_symbol_candidates_source_first(row, top_k=10, source_only=True)
pd.DataFrame(clean_tfidf_test)[["id", "tail", "score"]]

Cell 25 — Clean source-first TF-IDF + graph expansion

In [ ]:
# Cell 25 — Clean source-first TF-IDF + graph expansion

def hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=25, neighbor_bonus=0.35):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    score_map, seeds = tfidf_symbol_candidates_source_first(
        row,
        top_k=seed_top_k,
        source_only=True,
        return_score_map=True,
    )
    if not seeds:
        return []

    combined = {}

    for item in seeds:
        combined[item["id"]] = float(item["score"])

    for item in seeds:
        nid = item["id"]
        seed_score = float(item["score"])
        for nb in graph_index["adj"].get(nid, []):
            nb_node = graph_index["node_by_id"].get(nb)
            if nb_node is None:
                continue
            if is_test_node_id(nb):
                continue

            nb_base = float(score_map.get(nb, 0.0))
            total = nb_base + neighbor_bonus * seed_score

            # tiny repo-native bias
            prefix = repo_prefix_for_row(row)
            if prefix and nb.startswith(prefix):
                total += 0.05

            combined[nb] = max(combined.get(nb, 0.0), total)

    ranked = []
    for nid, score in combined.items():
        node = graph_index["node_by_id"].get(nid)
        if node is None:
            continue
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(score, 4),
            "preview": node["preview"],
        })

    ranked = sorted(ranked, key=lambda x: (-x["score"], x["id"]))
    return ranked[:final_top_k]

clean_hybrid_eval_rows = []
for _, row in subset.iterrows():
    clean_tfidf_ranked = tfidf_symbol_candidates_source_first(row, top_k=10, source_only=True)
    clean_hybrid_ranked = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=25)

    clean_tfidf_ids = [x["id"] for x in clean_tfidf_ranked]
    clean_hybrid_ids = [x["id"] for x in clean_hybrid_ranked]

    clean_hybrid_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "clean_tfidf_hit@10": ranked_hits_gold_symbol(clean_tfidf_ids[:10], row["gold_symbols"]),
        "clean_hybrid_hit@10": ranked_hits_gold_symbol(clean_hybrid_ids[:10], row["gold_symbols"]),
        "clean_hybrid_hit@25": ranked_hits_gold_symbol(clean_hybrid_ids[:25], row["gold_symbols"]),
    })

clean_hybrid_eval = pd.DataFrame(clean_hybrid_eval_rows)

print("=== CLEAN SOURCE-FIRST TF-IDF + GRAPH RESULTS ===")
print(pd.DataFrame({
    "graph_expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "graph_expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
    "clean_tfidf_hit@10": [clean_hybrid_eval["clean_tfidf_hit@10"].mean()],
    "clean_hybrid_hit@10": [clean_hybrid_eval["clean_hybrid_hit@10"].mean()],
    "clean_hybrid_hit@25": [clean_hybrid_eval["clean_hybrid_hit@25"].mean()],
}).round(3))

Cell 26 — Inspect if the cleaned retriever fixed the earlier bad example

In [ ]:
# Cell 26 — Inspect cleaned hybrid on previous failure examples

inspect_ids = [
    "rich_4077",
    "fastapi_14455",
    "fastapi_14430",
    "fastapi_11194",
    "fastapi_14482",
]

for iid in inspect_ids:
    rows = subset[subset["instance_id"] == iid]
    if len(rows) == 0:
        continue
    row = rows.iloc[0]

    cands = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=12)

    print("\n" + "=" * 120)
    print("INSTANCE:", row["instance_id"])
    print("REPO:", row["repo"])
    print("GOLD SYMBOLS:", row["gold_symbols"])
    print("\nCLEAN QUERY:")
    print(clean_query_text(row)[:500])

    if len(cands) == 0:
        print("No candidates returned.")
        continue

    display(pd.DataFrame(cands)[["id", "tail", "score", "preview"]])

Cell 27 — Snapshot file catalog + symbol → file mapping

In [ ]:
# Cell 27 — Robust snapshot python-file catalog + symbol -> file mapping
import tarfile
from pathlib import Path
from collections import defaultdict

SNAPSHOT_FILE_CACHE = {}

def get_snapshot_path(row):
    return SNAPSHOTS_DIR / f"{row['instance_id']}.tgz"

def all_python_suffixes(path_str: str):
    """
    For an archive member path like:
      some/root/fastapi/applications.py
    generate:
      some/root/fastapi/applications.py
      root/fastapi/applications.py
      fastapi/applications.py
      applications.py
    """
    p = Path(path_str)
    parts = p.parts
    out = set()

    for i in range(len(parts)):
        suffix = Path(*parts[i:])
        if suffix.suffix == ".py":
            out.add(str(suffix))

    return out

def get_snapshot_py_files(row):
    """
    Returns a set of normalized .py file paths extracted from the snapshot tar.
    Includes all suffix variants so we can match module paths robustly.
    """
    tpath = get_snapshot_path(row)
    if tpath in SNAPSHOT_FILE_CACHE:
        return SNAPSHOT_FILE_CACHE[tpath]

    py_files = set()

    if not tpath.exists():
        SNAPSHOT_FILE_CACHE[tpath] = py_files
        return py_files

    with tarfile.open(tpath, "r:gz") as tf:
        for m in tf.getmembers():
            if not m.isfile():
                continue
            name = m.name
            if not name.endswith(".py"):
                continue
            py_files.update(all_python_suffixes(name))

    SNAPSHOT_FILE_CACHE[tpath] = py_files
    return py_files

def symbol_id_to_module_candidates(symbol_id: str):
    """
    Convert a graph symbol id like:
      fastapi.dependencies.utils.get_dependant
    into likely module file paths:
      fastapi/dependencies/utils.py
      fastapi/dependencies/utils/__init__.py
      fastapi/dependencies.py
      ...
    """
    parts = str(symbol_id).split(".")
    candidates = []

    # progressively shorten from full dotted name toward package/module root
    for k in range(len(parts) - 1, 0, -1):
        mod = "/".join(parts[:k])
        candidates.append(f"{mod}.py")
        candidates.append(f"{mod}/__init__.py")

    # dedupe preserve order
    seen = set()
    ordered = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            ordered.append(c)
    return ordered

def symbol_id_to_candidate_files(row, symbol_id):
    py_files = get_snapshot_py_files(row)
    raw_candidates = symbol_id_to_module_candidates(symbol_id)

    hits = []

    # exact match first
    for c in raw_candidates:
        if c in py_files:
            hits.append(c)

    if hits:
        # dedupe preserve order
        out = []
        seen = set()
        for h in hits:
            if h not in seen:
                seen.add(h)
                out.append(h)
        return out

    # fallback: suffix match against normalized snapshot files
    for c in raw_candidates:
        for f in py_files:
            if f == c or f.endswith("/" + c):
                hits.append(f)

    out = []
    seen = set()
    for h in hits:
        if h not in seen:
            seen.add(h)
            out.append(h)
    return out

def symbols_to_ranked_files(row, symbol_candidates, max_files=10):
    ranked_files = []
    seen_files = set()

    for item in symbol_candidates:
        sid = item["id"]
        score = item["score"]
        cands = symbol_id_to_candidate_files(row, sid)

        for f in cands:
            if f not in seen_files:
                seen_files.add(f)
                ranked_files.append({
                    "file": f,
                    "from_symbol": sid,
                    "score": score,
                })

        if len(ranked_files) >= max_files:
            break

    return ranked_files[:max_files]

# quick debug on a few known cases
for iid in ["rich_4077", "fastapi_14455", "fastapi_14962"]:
    rows = subset[subset["instance_id"] == iid]
    if len(rows) == 0:
        continue
    row = rows.iloc[0]
    print("\n" + "=" * 100)
    print("INSTANCE:", iid)
    print("GOLD FILES:", row["gold_files"])

    cands = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=8)
    print("Top symbol IDs:")
    for x in cands[:5]:
        print("-", x["id"])

    mapped = symbols_to_ranked_files(row, cands, max_files=8)
    print("Mapped files:")
    for m in mapped:
        print("-", m["file"], "| from:", m["from_symbol"])

Cell 28 — File recall@1/@3/@5 from improved hybrid retriever

In [ ]:
# Cell 28 — File-level recall evaluation

def ranked_hits_gold_file(ranked_files, gold_files):
    gold = set(gold_files or [])
    return any(f in gold for f in ranked_files)

file_eval_rows = []

for _, row in subset.iterrows():
    symbol_ranked = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=25)
    file_ranked = symbols_to_ranked_files(row, symbol_ranked, max_files=10)
    file_ids = [x["file"] for x in file_ranked]

    file_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "gold_files": row["gold_files"],
        "pred_files": file_ids,
        "file_hit@1": ranked_hits_gold_file(file_ids[:1], row["gold_files"]),
        "file_hit@3": ranked_hits_gold_file(file_ids[:3], row["gold_files"]),
        "file_hit@5": ranked_hits_gold_file(file_ids[:5], row["gold_files"]),
        "file_hit@10": ranked_hits_gold_file(file_ids[:10], row["gold_files"]),
        "num_pred_files": len(file_ids),
    })

file_eval = pd.DataFrame(file_eval_rows)

print("=== FILE-LEVEL RECALL FROM CLEAN HYBRID SYMBOL RETRIEVER ===")
print(pd.DataFrame({
    "file_hit@1": [file_eval["file_hit@1"].mean()],
    "file_hit@3": [file_eval["file_hit@3"].mean()],
    "file_hit@5": [file_eval["file_hit@5"].mean()],
    "file_hit@10": [file_eval["file_hit@10"].mean()],
    "avg_pred_files": [file_eval["num_pred_files"].mean()],
}).round(3))

file_eval.head(10)

Cell 29 — Inspect file-level misses

In [ ]:
# Cell 29 — Inspect file-level failures

file_fail = file_eval[~file_eval["file_hit@5"]].copy()

print("File-level misses at @5:", len(file_fail))
display(file_fail[["instance_id", "repo", "gold_files", "pred_files"]].head(10))

for iid in file_fail["instance_id"].head(5):
    row = subset[subset["instance_id"] == iid].iloc[0]
    symbol_ranked = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=12)
    file_ranked = symbols_to_ranked_files(row, symbol_ranked, max_files=8)

    print("\n" + "=" * 120)
    print("INSTANCE:", row["instance_id"])
    print("REPO:", row["repo"])
    print("GOLD FILES:", row["gold_files"])
    print("\nISSUE SNIPPET:")
    print((row["problem_statement"] or "")[:700])

    print("\nSYMBOL CANDIDATES:")
    display(pd.DataFrame(symbol_ranked)[["id", "tail", "score"]])

    print("\nPREDICTED FILES:")
    display(pd.DataFrame(file_ranked))

Cell 30 — Inspect sample submission

In [ ]:
sample_files = sorted([p for p in SAMPLE_SUBMISSION.rglob("*") if p.is_file()])
print("Sample submission file count:", len(sample_files))
for p in sample_files:
    print("-", p.relative_to(SAMPLE_SUBMISSION))

Cell 31 — Read key sample submission files

In [ ]:
key_files = [
    SAMPLE_SUBMISSION / "agent.yaml",
    SAMPLE_SUBMISSION / "eval_config.yaml",
    SAMPLE_SUBMISSION / "configs" / "sampling.yaml",
    SAMPLE_SUBMISSION / "prompts" / "system.md",
    SAMPLE_SUBMISSION / "prompts" / "analyzer.md",
    SAMPLE_SUBMISSION / "sub_agents" / "code_analyzer.yaml",
]

for p in key_files:
    print("" + "=" * 100)
    print(p.relative_to(SAMPLE_SUBMISSION))
    print(p.read_text(encoding="utf-8", errors="ignore")[:4000])

Cell 32 — Final improved system prompt

In [ ]:
hot_fastapi = hot_files[hot_files["repo"] == "fastapi/fastapi"].head(5)["gold_files"].tolist()
hot_rich = hot_files[hot_files["repo"] == "Textualize/rich"].head(5)["gold_files"].tolist()
hot_requests = hot_files[hot_files["repo"] == "psf/requests"].head(5)["gold_files"].tolist()
hot_httpx = hot_files[hot_files["repo"] == "encode/httpx"].head(5)["gold_files"].tolist()

FINAL_SYSTEM_PROMPT = f"""
You are an offline software-engineering agent operating inside the Gemma 4 Developer Agent competition harness.

Your goal is to resolve the issue by producing the smallest correct source patch.

General policy:
1. Restate the bug in 1–3 sentences.
2. Localize before editing.
3. Prefer code-graph evidence, semantic similarity, and nearby symbols over broad wandering.
4. Read only the most relevant files and symbols.
5. Form a concrete defect hypothesis before making edits.
6. Prefer minimal, behavior-preserving fixes.
7. Avoid unrelated refactors, cleanup, or style-only edits.
8. Verify narrowly when practical.
9. If confidence remains low after bounded search, do not make random edits.

Search policy:
- Start from issue keywords, symbol names, and behavioral clues.
- Use graph-based search to identify candidate symbols.
- Expand neighbors around promising symbols.
- Convert promising symbols into likely source files.
- Prioritize source files over test files.
- Prefer source modules over __init__ files unless evidence strongly suggests otherwise.

Editing policy:
- Preserve interfaces unless the issue clearly requires a change.
- Reuse nearby implementation patterns.
- Make the smallest patch that fully fixes the issue.
- Keep temporary debugging artifacts out of the final patch.

Verification policy:
- Prefer focused reproductions or narrow tests over broad expensive runs.
- If a targeted test exists, use it first.
- Stop once you have the highest-value evidence available within budget.

Observed public-task priors:
- Most fixes are small.
- Many fixes touch exactly one source file.
- Many issues do not directly name the right file, so localization matters.
- Neighbor expansion in code graphs can help recover the correct symbol.
- Source-file prioritization matters.

Public-repo hotspot examples from training data:
- fastapi/fastapi: {hot_fastapi}
- Textualize/rich: {hot_rich}
- psf/requests: {hot_requests}
- encode/httpx: {hot_httpx}

Do not overfit to these hotspots; use them only as weak priors.

Final behavior:
- Submit only the best bounded patch you can justify.
- Prefer no patch over low-confidence random edits.
""".strip()

print(FINAL_SYSTEM_PROMPT[:4000])

Cell 33 — Final analyzer prompt

In [ ]:
FINAL_ANALYZER_PROMPT = """
You are a code-analysis specialist.

Your job:
1. Summarize the issue clearly.
2. Identify the most likely symbols involved.
3. Identify the most likely source files involved.
4. Prefer source files over tests.
5. Explain why each candidate is relevant.
6. Avoid speculative large-scale exploration.

Return concise, actionable localization guidance for the patching agent.
""".strip()

print(FINAL_ANALYZER_PROMPT)

Cell 34 — apply-only version

In [ ]:
import shutil

SUBMISSION_DIR = WORKING / "submission"
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)

shutil.copytree(SAMPLE_SUBMISSION, SUBMISSION_DIR)
print("Copied sample submission to:", SUBMISSION_DIR)

Cell 35 — Overwrite prompt files

In [ ]:
system_prompt_path = SUBMISSION_DIR / "prompts" / "system.md"
analyzer_prompt_path = SUBMISSION_DIR / "prompts" / "analyzer.md"

system_prompt_path.write_text(FINAL_SYSTEM_PROMPT, encoding="utf-8")
print("Updated:", system_prompt_path)

if analyzer_prompt_path.exists():
    analyzer_prompt_path.write_text(FINAL_ANALYZER_PROMPT, encoding="utf-8")
    print("Updated:", analyzer_prompt_path)
else:
    print("Analyzer prompt file not found; skipping")

Cell 36 — Inspect working YAML files

In [ ]:
config_files = sorted([p for p in SUBMISSION_DIR.rglob("*") if p.suffix.lower() in {".yaml", ".yml", ".md"}])

for p in config_files:
    print("" + "=" * 100)
    print(p.relative_to(SUBMISSION_DIR))
    print(p.read_text(encoding="utf-8", errors="ignore")[:4000])

 Cell 37 — Optional YAML edits

In [ ]:
# Recommended final adjustments before rebuilding submission.zip

EVAL_CONFIG_TEXT = """# Optional participant evaluation configuration for Stage 1 inference.
# Controls per-task execution budgets and sandbox command timeouts.
evaluation:
  timeout_seconds: 120
  max_tool_calls: 16
  max_time_minutes: 4
  max_turns: 60
"""

eval_config_path = SUBMISSION_DIR / "eval_config.yaml"
eval_config_path.write_text(EVAL_CONFIG_TEXT, encoding="utf-8")
print("Updated:", eval_config_path)
print(eval_config_path.read_text(encoding="utf-8"))

FINAL_SYSTEM_PROMPT = FINAL_SYSTEM_PROMPT.replace(
    "Final behavior:\n- Submit only the best bounded patch you can justify.\n- Prefer no patch over low-confidence random edits.",
    "Final behavior:\n- Submit the best bounded patch you can justify.\n- If confidence is imperfect, prefer the smallest evidence-backed source patch over broad wandering or random edits.\n- Once you have your best bounded fix, call submit_patch."
)

system_prompt_path.write_text(FINAL_SYSTEM_PROMPT, encoding="utf-8")
print("\nUpdated:", system_prompt_path)
print(system_prompt_path.read_text(encoding="utf-8")[-800:])

Cell 38 — Build submission.zip

In [ ]:
import zipfile

SUBMISSION_ZIP = WORKING / "submission.zip"
if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(SUBMISSION_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(SUBMISSION_DIR))

print("Created:", SUBMISSION_ZIP)
print("Size (MB):", round(SUBMISSION_ZIP.stat().st_size / (1024 * 1024), 3))

Cell 39 — Sanity check + final status

In [ ]:
import zipfile

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    names = zf.namelist()

print("Archive entries:", len(names))
print("First 50 entries:")
for name in names[:50]:
    print("-", name)

assert "agent.yaml" in names, "Expected agent.yaml at archive root"
print("Sanity check passed: agent.yaml is present at zip root.")

print("\n=== READY STATUS ===")
print("Submission bundle built:", SUBMISSION_ZIP.exists())
print("Submission path:", SUBMISSION_ZIP)

cell 40

In [ ]:
import shutil

SUBMISSION_DIR = WORKING / "submission"
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
shutil.copytree(SAMPLE_SUBMISSION, SUBMISSION_DIR)

adapters = SUBMISSION_DIR / "adapters"
if adapters.exists():
    shutil.rmtree(adapters)

for path in SUBMISSION_DIR.rglob("*.yaml"):
    lines = path.read_text(encoding="utf-8").splitlines(keepends=True)
    kept = [line for line in lines if not line.strip().startswith("adapter:")]
    path.write_text("".join(kept), encoding="utf-8")

print("adapter keys left", [
    str(p.relative_to(SUBMISSION_DIR))
    for p in SUBMISSION_DIR.rglob("*.yaml")
    if "adapter:" in p.read_text(encoding="utf-8")
])
print("adapters dir exists", (SUBMISSION_DIR / "adapters").exists())


cell 41

In [ ]:
(SUBMISSION_DIR / "prompts" / "system.md").write_text("""You are an autonomous software engineer. Resolve the issue under /workspace and stop.

## Budget
You have at most 20 tool calls and 8 minutes. Do not explore the repository. Find the symbol, edit it, run one targeted test, submit.

## Locate
- Take file names, function names, class names, and exception types from the problem statement and hints.
- If a symbol name is present, call search_similar_code with that symbol name only. Do not pass a sentence. The search index matches symbol names, not descriptions.
- Use get_code_neighbors on the symbol the search returns. If the tool errors or the symbol is an async function and is missing, fall back to run_command with a single rg call, for example: rg -n "symbol_name" -g "*.py" -g "!tests/**"
- read_file at most 150 lines at a time. Pass start_line and end_line. Never read a whole file in one call.

## Edit
- Edit one source file. Open a second source file only if the targeted test names another symbol.
- Edit library code only. Never create, modify, or delete anything under tests/, any test_*.py, or any *_test.py.
- Never modify /workspace/pytest.ini or /workspace/conftest.py.
- Use edit_file with a small unique old_string. If the tool call would be long, split the change into several edits.
- Put scratch scripts in /tmp, never in /workspace. Example: /tmp/repro.py. Delete any scratch file you created under /workspace before submit_patch.

## Test
- Run one targeted test file, not the suite. Example: python3 -m pytest tests/test_mod.py -k test_name -q
- Never run bare pytest, pytest ., or unittest discover.
- If a failure is in a file you did not mean to change, revert that file. Do not repair pre-existing failures.

## Submit
- Call submit_patch only after the targeted test has been run, and only as the last tool call.
- If you changed a library file, submit that diff. An empty diff is not a completed task. Do not edit just to produce a diff.
- submit_patch does not spend a tool call. get_status does not spend a tool call.
- After submit_patch returns, reply with one sentence and stop. Do not edit again.
""", encoding="utf-8")

(SUBMISSION_DIR / "prompts" / "analyzer.md").write_text("""You locate code. You do not edit it.

1. Extract symbol names from the task. Call search_similar_code with a symbol name, not a sentence.
2. Call get_code_neighbors on the best match.
3. read_file the matching file in a 150-line window.
4. Reply with file paths, symbol names, and the line range to change. Do not propose a whole-file rewrite.
""", encoding="utf-8")

(SUBMISSION_DIR / "eval_config.yaml").write_text("""evaluation:
  timeout_seconds: 120
  max_tool_calls: 20
  max_time_minutes: 8
  max_turns: 30
""", encoding="utf-8")

(SUBMISSION_DIR / "configs" / "sampling.yaml").write_text("""temperature: 0.2
top_p: 0.95
top_k: 64
max_output_tokens: 8192
thinking_config:
  thinking_level: medium
  thinking_budget: 2048
  include_thoughts: true
""", encoding="utf-8")

print("prompts written")

Cell 42

In [ ]:
import zipfile

SUBMISSION_ZIP = WORKING / "submission.zip"
if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(SUBMISSION_DIR.rglob("*")):
        if path.is_file() and "adapters" not in path.parts:
            zf.write(path, path.relative_to(SUBMISSION_DIR).as_posix())

with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
    system = zf.read("prompts/system.md").decode("utf-8")
    agent = zf.read("agent.yaml").decode("utf-8")

assert "agent.yaml" in names
assert not any(name.startswith("adapters/") for name in names)
assert "adapter:" not in agent
assert "gemma-4-31b-it-qat-w4a16-ct" in agent
assert "one source file" in system
assert "fastapi" not in system.lower()
assert "semantic similarity" not in system.lower()
print("entries", len(names))
print("\n".join(names))
print("bytes", SUBMISSION_ZIP.stat().st_size)


Cell 43 — stop unless this is an L4

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU L4 and restart. Do not continue."
name = torch.cuda.get_device_name(0)
mib = torch.cuda.get_device_properties(0).total_memory // (1024 * 1024)
print("devices", torch.cuda.device_count())
print("gpu0", name)
print("gpu0_mib", mib)
assert mib >= 20000, f"{name} has {mib} MiB. This serve needs an L4. Stop."


Cell 44 — install the wheelhouse, no venv

In [ ]:
# Cell 44 (optimized) — wheelhouse install plus cp313 fill, offline-first, with phase timing.
# Attach the wheelhouse always; attach a cp313 wheel pack if the session has no internet.
import glob
import importlib
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
os.environ["VLLM_ENGINE_READY_TIMEOUT_S"] = "1200"
os.environ["VLLM_NO_USAGE_STATS"] = "1"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

FORCE_REINSTALL = False   # True re-unpacks all 824 MB of wheelhouse wheels on every run
PIP_FAST_FAIL = ["--retries", "1", "--timeout", "10"]   # a no-internet session fails in seconds

T0 = time.time()


def mark(label):
    print(f"[{time.time() - T0:6.1f}s] {label}")


print("python", sys.version.split()[0], "|", sys.executable)

INPUT = Path("/kaggle/input")
candidates = [p for p in INPUT.rglob("*") if p.is_dir() and any(p.glob("swegemma-*.whl"))]
assert candidates, (
    "Wheelhouse is not attached. Add Input -> Datasets -> "
    "metric/gemma-4-developer-agent-wheelhouse, then restart. Do not continue."
)
WHEELHOUSE_DIR = sorted(candidates)[0]
mark(f"wheelhouse {WHEELHOUSE_DIR} ({len(list(WHEELHOUSE_DIR.glob('*.whl')))} wheels)")

SUPPORTED = set(sys_tags())

# One scan of every attached input collects wheels this interpreter can use, keyed by dist/version.
# The wheelhouse's cp312-only wheels can never match a 3.13 interpreter, so only genuinely
# compatible extras (for example a cp313-fill-wheels dataset) end up in this map.
offline = {}
for path in INPUT.rglob("*.whl"):
    try:
        dist, version, _, wheel_tags = parse_wheel_filename(path.name)
    except Exception:
        continue
    if wheel_tags & SUPPORTED:
        offline.setdefault((dist, version), str(path))
mark(f"input scan: {len(offline)} wheels this interpreter can use are available in /kaggle/input")

for pattern in (
    "/usr/local/lib/python*/dist-packages/*cutlass*.pth",
    "/usr/local/lib/python*/site-packages/*cutlass*.pth",
):
    for pth in glob.glob(pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# /tmp survives the whole session. Symlinks from an earlier run would otherwise keep pinning the
# previous wheelhouse version (swegemma 0.2.7 / adk_submission 0.2.12) after the dataset updates.
tmp_whl = Path("/tmp/wheelhouse")
shutil.rmtree(tmp_whl, ignore_errors=True)
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    os.symlink(w, tmp_whl / name)

wheels, skipped = [], []
for w in sorted(tmp_whl.glob("*.whl")):
    (wheels if parse_wheel_filename(w.name)[3] & SUPPORTED else skipped).append(w)

install_cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-deps"]
if FORCE_REINSTALL:
    install_cmd.append("--force-reinstall")
subprocess.run([*install_cmd, *[str(w) for w in wheels]], check=True)
mark(f"wheelhouse install done: {len(wheels)} wheels installed, {len(skipped)} skipped for this interpreter")

# The skipped wheels are CPython 3.12-only builds. Prefer an attached cp313 copy (no network, no
# retries); fall back to PyPI, which needs Internet ON in Session options. --no-deps is required:
# xgrammar declares transformers<5 while the wheelhouse pins transformers 5.13.1.
unresolved = []
for w in skipped:
    dist, version, _, _ = parse_wheel_filename(w.name)
    local = offline.get((dist, version))
    if local:
        proc = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", local],
            capture_output=True,
            text=True,
        )
        mark(f"{dist}=={version} from attached wheels: {'ok' if proc.returncode == 0 else 'FAILED'}")
        if proc.returncode != 0:
            unresolved.append(f"{dist}=={version}")
        continue
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", *PIP_FAST_FAIL, f"{dist}=={version}"],
        capture_output=True,
        text=True,
    )
    mark(f"{dist}=={version}: {'installed from PyPI' if proc.returncode == 0 else 'unavailable'}")
    if proc.returncode != 0:
        unresolved.append(f"{dist}=={version}")

importlib.invalidate_caches()

# Smoke-test the serving stack in a fresh interpreter so a broken import cannot hide behind
# kernel state, and fail here rather than 340 seconds into a server start.
probe = subprocess.run(
    [
        sys.executable,
        "-c",
        "import vllm, xgrammar, msgspec, pybase64, cbor2, ijson, setproctitle, outlines_core;"
        "print('vllm', vllm.__version__)",
    ],
    capture_output=True,
    text=True,
)
mark("import probe: " + (probe.stdout or probe.stderr).strip()[-300:])
assert probe.returncode == 0, (
    "Serving stack incomplete on Python " + sys.version.split()[0]
    + ". Unresolved: " + (", ".join(unresolved) or "none reported")
    + ". Three ways out: (1) attach a cp313 wheel pack to /kaggle/input and re-run; "
    + "(2) Session options -> Internet -> On and re-run; "
    + "(3) run the notebook on the Docker image the official Getting Started notebook pins "
    + "(gcr.io/kaggle-private-byod/python@sha256:37c64f7dd9c54116ecd1bcc88817c5469b88387388fade02bfa8bf3fc647d461), "
    + "where nothing is skipped. Do not start the server."
)

import swegemma  # noqa: F401
import adk_submission  # noqa: F401
import adk_eval_core  # noqa: F401

mark("wheelhouse installation complete; ready for Cell 46")

cell44b_torch_pin

In [ ]:
# Cell 44b — match torch to what vllm 0.19.1 compiled against, installed from an attached wheel dataset.
# L4 x4 has no internet in this competition, so nothing in this cell touches PyPI.
import importlib.metadata as md
import subprocess
import sys
from pathlib import Path

from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

INPUT = Path("/kaggle/input")
TORCH_PIN = "2.10.0"
WANT = [("torch", "2.10.0"), ("torchvision", "0.25.0"), ("torchaudio", "2.10.0")]
SUPPORTED = set(sys_tags())


def probe(code):
    return subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)


def base_version(v):
    return None if v is None else str(v).split("+")[0]


def image_cuda_tag():
    """cu128 for torch.version.cuda == 12.8, so a mismatched CUDA pack cannot be picked by accident."""
    r = probe("import torch; print(torch.version.cuda)")
    cuda = r.stdout.strip() if r.returncode == 0 else ""
    return "cu" + cuda.replace(".", "") if cuda and cuda != "None" else None


def local_wheel(dist, version):
    want_cu = image_cuda_tag()
    hits = []
    for p in INPUT.rglob("*.whl"):
        try:
            d, v, _, tags = parse_wheel_filename(p.name)
        except Exception:
            continue
        if d == dist and base_version(v) == version and (tags & SUPPORTED):
            hits.append(p)
    if not hits:
        return None
    hits.sort(key=str)
    if want_cu:
        preferred = [p for p in hits if want_cu in p.name]
        if preferred:
            hits = preferred
    print(f"    {dist}=={version}: using {hits[-1].name}")
    return str(hits[-1])


try:
    have = md.version("torch")
except md.PackageNotFoundError:
    have = None
print("torch on disk:", have, "| required:", TORCH_PIN, "| image cuda tag:", image_cuda_tag())

if base_version(have) != TORCH_PIN:
    plan = []
    for dist, version in WANT:
        w = local_wheel(dist, version)
        if w is None:
            print(f"    MISSING from /kaggle/input: {dist}=={version}")
        else:
            plan.append(w)
    if len(plan) != len(WANT):
        raise RuntimeError(
            "Attach the torch cp313 wheel dataset (torch 2.10.0 + torchvision 0.25.0 + torchaudio 2.10.0), "
            "then re-run this cell. L4 x4 cannot reach PyPI, so the wheels must arrive as an input."
        )
    proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *plan],
                          capture_output=True, text=True)
    print("pip exit", proc.returncode)
    if proc.returncode != 0:
        print((proc.stderr or proc.stdout or "").strip()[-600:])
        raise RuntimeError("offline torch install failed")
    print("torch installed on disk. RESTART THE KERNEL, then re-run Cell 44, this cell, Cell 45 and Cell 46.")
else:
    r = probe("import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())")
    print("probe torch:", (r.stdout or r.stderr).strip()[-200:])

# The precise test for the error we are fixing: vllm's compiled extension links against libtorch.
r = probe("import vllm._C; print('vllm._C ok')")
print("vllm._C:", (r.stdout or r.stderr).strip()[-300:])
if r.returncode != 0:
    raise RuntimeError(
        "vllm._C still fails to load. If the message names a torch symbol (c10::...), the kernel is still "
        "using the pre-install torch: restart the kernel and re-run. If it names a CUDA/nvidia library, the "
        "attached torch build does not match the image's CUDA runtime - attach the pack matching "
        "torch.version.cuda (" + str(image_cuda_tag()) + ") instead."
    )

# vllm 0.19.1 also requires nvidia-cutlass-dsl, which the wheelhouse does not carry.
r = probe("import nvidia_cutlass_dsl; print('nvidia_cutlass_dsl ok')")
if r.returncode != 0:
    w = local_wheel("nvidia-cutlass-dsl", "4.8.0")
    if w:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", w],
                       capture_output=True, text=True)
        r = probe("import nvidia_cutlass_dsl; print('nvidia_cutlass_dsl ok')")
    print("nvidia_cutlass_dsl:", (r.stdout or r.stderr).strip()[-200:])

print("cell 44b done — ready for Cell 45")

Cell 44c - restore the submission from the attached dataset

In [ ]:
# Copy the attached submission dataset into the working dir. No pasting, no 8 KB cell.
import shutil
from pathlib import Path

hits = [p for p in Path("/kaggle/input").rglob("sub_agents/locate.yaml")]
assert hits, "Attach the gemma4-dev-submission dataset first"
SRC = hits[0].parents[1]
TARGET = Path("/kaggle/working/submission")
if TARGET.exists():
    shutil.rmtree(TARGET)
shutil.copytree(SRC, TARGET)
files = sorted(p.relative_to(TARGET).as_posix() for p in TARGET.rglob("*") if p.is_file())
print("wrote", len(files), "files to", TARGET)
print(files)
root = (TARGET / "agent.yaml").read_text(encoding="utf-8")
assert "SequentialAgent" in root and "sub_agents/locate.yaml" in root
print("restore ok")

Cell 44d - one-off bridge: write the C1 locate.md into the live session.

In [ ]:
# Cell 44d (C5) - install the C5 locate.md in the live session.
# C5: one-liners cannot hold a decorator or a class body (use exec), and two failed reproductions are enough.
from pathlib import Path

PROMPTS = Path("/kaggle/working/submission/prompts")
C5_LOCATE = '''Find the library file and the symbol. Do not edit.

- Search only a word that appears in the issue. Do not invent a symbol name. If the issue has two words for one name, join them with an underscore.
- Call search_similar_code with that word only. Do not pass a sentence. Discard a result under tests. Use the first library symbol.
- If the issue writes a name followed by parentheses, that name is the symbol. A search hit with a different name is not that symbol. Call run_command with grep -n -r that name once. Open the library file that defines it. One read of that definition, then end.
- If that search returns nothing, stop searching. Reply with the file path if you already opened it. Do not call rg.
- read_file at most 150 lines. Pass start_line and end_line.
- Before you end, reproduce the issue with one minimal `python3 -c "..."` command; do not create a file.
- run_command is read-only here: grep, git log, ls, cat, sed -n, and python3 -c that only prints. Never write: no redirect, no sed -i, no python that opens a file for writing, no edit_file. This stage must leave the repository byte-identical.
- One `python3 -c` line cannot hold a decorator, a class body, or any multi-line block: wrap that code in `exec("...")` with \n separators and quote the whole command with single quotes. If the reproduction fails twice, stop reproducing and report the file, symbol and range you already have.
- The file you report must be on the path that command executes: if it raises, the last repository frame; if the output is wrong, follow the functions it called until behaviour diverges. Not a guess from names. If one command cannot show it, keep the file you already found.
- End with the library path, the symbol, and that start_line and end_line. Do not create a file.'''

target = PROMPTS / "locate.md"
before = target.read_text(encoding="utf-8") if target.is_file() else ""
target.write_text(C5_LOCATE, encoding="utf-8")
now = target.read_text(encoding="utf-8")
print(f"locate.md: {len(before)} -> {len(now)} chars")
print("exec line:", "cannot hold a decorator" in now)
print("two-strike line:", "fails twice" in now)
print("read-only line still present:", "run_command is read-only here" in now)
assert "cannot hold a decorator" in now and "fails twice" in now
assert "run_command is read-only here" in now
print("C5 locate.md installed")

Cell 44e - install D1-D4 into the live session.

In [ ]:
# Cell 44e - install D1-D4 into the live session.
# D1 deadline-aware locate (+ free get_status), D2 graph tools dropped, D3 eval timeout 300,
# D4 the two dead files (prompts/system.md, sub_agents/code_analyzer.yaml) removed.
from pathlib import Path

AGENT_DIR = Path("/kaggle/working/submission")

FILES = {
    "prompts/locate.md": '''Find the library file and the symbol. Do not edit.

- **The clock decides this task, not the call count: 30 calls at roughly 10 seconds each is the whole session.** This stage gets about 120 seconds. If 120 seconds have elapsed, stop and report the best file, symbol and range you already have — a perfect answer delivered after the session dies scores zero.
- First call the repo-scout skill: run the script skills/repo-scout/scripts/locate.py with the problem statement as its argument (see the skill's SKILL.md). It returns the ten most likely files with the lines that matched, fused from path, identifier, code and error-string rankings. If the skill call fails for any reason, do not retry it: fall back to grep immediately.
- The skill's list is a ranking of candidates, not a verdict. Read its evidence lines and commit to the one candidate whose evidence contains a word the issue names. Do not drift to a later candidate on a hunch: measured on the public tasks, the model picks correctly 67% of the time from this list against 47% for the list's own first entry, and most of its wrong answers are candidates it talked itself into.
- You have read_file, run_command and search_similar_code. There is no graph tool here. Every call you spend is a call the edit stage cannot use.
- You must have chosen the file by call 8 and reported it by call 12, and in any case before 120 seconds elapse. If you cannot, report the best candidate you already have: a wrong edit can be corrected, no edit cannot.
- Search only a word that appears in the issue. Do not invent a symbol name. If the issue has two words for one name, join them with an underscore.
- If the issue names a class or type and a method, search the class or type word first: a method search returns classes that already have that method, and the class named in the issue is the one missing it.
- Call run_command with grep -n -r "<word>" --include="*.py" . once per word. Discard a result under tests. Use the first library hit. If the issue is a title with no symbol, grep each content word once and keep the file whose path matches the most words.
- If the issue writes a name followed by parentheses, that name is the symbol. A search hit with a different name is not that symbol.
- If a search returns nothing, stop searching that word. Do not call rg.
- read_file at most 150 lines. Pass start_line and end_line.
- Reproduction is optional and expensive: run one minimal `python3 -c "..."` command only when fewer than 90 seconds have elapsed and the candidate file is already certain — on a 500-file repository a single import can cost 20 seconds. Never let it be your last action; do not create a file.
- Never leave a scratch file under /workspace: the patch is captured with `git add -N . && git diff HEAD`, so any untracked file there ships to the grader. If you must write a script, write it to /tmp.
- One `python3 -c` line cannot hold a decorator, a class body, or any multi-line block: wrap that code in `exec("...")` with \n separators and quote the whole command with single quotes. If the reproduction fails twice, stop reproducing and report the file, symbol and range you already have.
- run_command is read-only here: grep, git log, ls, cat, sed -n, and python3 -c that only prints. Never write: no redirect, no sed -i, no python that opens a file for writing. This stage must leave the repository byte-identical.
- The file you report must be on the path that command executes: if it raises, the last repository frame; if the output is wrong, follow the functions it called until behaviour diverges. Not a guess from names. If one command cannot show it, keep the file you already found.
- End with the library path, the symbol, and that start_line and end_line. Do not create a file.
''',
    "prompts/edit.md": '''Edit the library file named in {locate_report}. Do not search. Do not submit.

- Read only the start_line to end_line named in {locate_report}. Do not read from line 1. Call edit_file on that range.
- Use edit_file with a small unique old_string. Do not create a file.
- You have exactly two tools: read_file and edit_file. You cannot run commands, tests, or shell scripts; do not try. The reproduction in {locate_report} was already run by the previous stage.
- Never create, modify, or delete anything under tests/, any test_*.py, or any *_test.py.
- Never modify /workspace/pytest.ini or /workspace/conftest.py.
- Make your edit on your first or second call; the locate stage already spent most of the budget.
- If a file you did not intend to change is now modified, restore it before you finish.
- The harness grades the working tree as it stands, so a correct edit made in time counts even if nothing else happens. Do not edit just to produce a diff, but an empty diff is not a completed task.
''',
    "prompts/verify.md": '''Run one targeted test, then submit. Do not edit.

- Run one targeted test file, not the suite. Example: python3 -m pytest tests/test_mod.py -k test_name -q
- Never create or modify a test file, anything under tests/, or pytest.ini/conftest.py. The grader resets those paths and a stray test file can make the whole task fail. Use `python3 -c` for anything you need to run.
- Never run bare pytest, pytest ., or unittest discover.
- Call submit_patch only after that test has been run, and only as the last tool call.
- If git diff is empty, do not call submit_patch.
- submit_patch does not spend a tool call and still works when the tool-call budget is exhausted. After it returns, reply with one sentence and stop.
- Call get_status first: it costs no tool call. If the time remaining is under 120 seconds, skip the test and submit immediately — the grader grades the patch, and a session that dies inside pytest leaves nothing verified. Under 45 seconds, submit without any further call at all.
- When you do run a test, make it one narrow node id (`-x -q -k name`) so it cannot eat the remaining clock. On a large repository a bare test file can take a minute.
- If a failure is in a file you did not mean to change, revert that file before submitting. Do not repair pre-existing failures.
- The patch is whatever `git add -N . && git diff HEAD` sees, so anything untracked under /workspace ships to the grader, and the harness has already committed its own pytest.ini and conftest.py. Delete any scratch file you created and keep your reproduction in /tmp or in `python3 -c`.
- submit_patch returns `files_changed`. If that number is larger than the number of library files you edited, an extra file is riding along: check `git diff --name-only`, revert the extras, and submit again.
''',
    "sub_agents/locate.yaml": '''name: locate_agent
description: Finds the one library file, symbol and line range the issue names. Never edits.
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include ../prompts/locate.md
output_key: locate_report
disallow_transfer_to_parent: true
disallow_transfer_to_peers: true
tools:
  - read_file
  - run_command
  - search_similar_code
skills:
  - skills/repo-scout
generate_content_config: !include ../configs/sampling.yaml
''',
    "sub_agents/verify.yaml": '''name: verify_agent
description: Runs one targeted test and submits the patch. Does not edit.
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include ../prompts/verify.md
disallow_transfer_to_parent: true
disallow_transfer_to_peers: true
tools:
  - run_command
  - submit_patch
  - get_status
generate_content_config: !include ../configs/sampling.yaml
''',
    "eval_config.yaml": '''# The host DOES read per-task limits from here ("You may, but are not required, to set per-task time
# limits in your submission's eval_config.yaml") and the whole run must fit 12 hours including
# sandbox setup. 120 tasks x 4.5 min = 540 min of agent time, leaving ~90 s/task for setup.
evaluation:
  timeout_seconds: 300
  max_tool_calls: 30
  max_time_minutes: 4.5
  max_turns: 60
''',
    "configs/sampling.yaml": '''temperature: 0.2
top_p: 0.95
top_k: 64
max_output_tokens: 8192
thinking_config:
  thinking_level: medium
  thinking_budget: 2048
  include_thoughts: true
''',
    "skills/repo-scout/SKILL.md": '''---
name: repo-scout
description: Rank repository files for a bug report before any file is opened. Use for every task, especially when the issue names no file or symbol. Returns the top 10 candidate files with the evidence lines that matched.
---

# repo-scout

Deterministic lexical retrieval over the repository. Six BM25 rankings are fused so that a file which
leads in any single ranking cannot be buried by files that are merely average in all of them:

1. **path** - the issue's words against file paths, so `text.py` scores for a markdown-styling issue.
2. **definition** - CamelCase and snake_case names taken from the issue against the names of the symbols
   each file defines (parsed with `ast`).
3. **code** - the issue's words against every identifier in the file.
4. **symbol** - the issue's words against each function or class body, kept at symbol resolution so a
   small function in a large file is not diluted; the winning symbol and its line range are reported.
5. **literal** - backticked or quoted strings from the issue against strings in the file.
6. **directory** - the issue's words against directory and module names, so `docs_src/<feature>/` scores.

Fusion: reciprocal rank fusion with k=10, plus **twice the best single-ranking contribution** per file.

## How to use it

Run the script once, before any grep:

    python3 scripts/locate.py "<problem statement>"

Add `--top 20` for a wider list, or `--include-tests` only if you suspect the fix is in a test.
It prints JSON: `candidates[0]` is the best file, with `symbol`, `lines` and `evidence` showing what
matched. Read the top candidate's evidence, confirm the symbol with one `grep -n`, then report the file,
symbol and range.

Measured on the 129 public tasks: the gold file is in the top-10 for **83.7%** of tasks and first for
**44.2%**. The script only reads files: it never writes, never calls the network, and needs no third-party
packages.
''',
    "skills/repo-scout/scripts/locate.py": '''#!/usr/bin/env python3
"""repo-scout v2: rank repository files (and symbols) for an issue.

Pure standard library, no writes, no network. Runs inside the competition sandbox through
run_skill_script, or standalone:

    python3 locate.py "<issue text>" [--root /workspace] [--top 10] [--include-tests]

Six rankings are fused with reciprocal rank fusion:
  1. path        issue words against file paths
  2. definition  issue identifiers against the names of symbols each file defines (ast)
  3. code        issue words against every identifier in the file
  4. symbol      issue words against each function/class body, kept at symbol resolution
  5. literal     backticked and quoted strings from the issue against strings in the file
  6. mention     files whose path, stem or dotted module is named verbatim in the issue
Output is JSON: candidates with path, score, symbol, lines and evidence lines.
"""

from __future__ import annotations

import argparse
import ast
import json
import math
import os
import sys
from collections import Counter
from pathlib import Path

STOP = {
    "the", "and", "for", "with", "that", "this", "from", "have", "has", "was", "were", "are", "not",
    "but", "you", "your", "when", "which", "will", "would", "there", "their", "then", "than", "them",
    "into", "does", "doesn", "isn", "use", "using", "used", "can", "cannot", "should", "must", "may",
    "fix", "fixes", "issue", "bug", "error", "add", "support", "update", "new", "one", "two", "all",
    "any", "see", "name", "test", "tests", "python", "code", "file", "files", "line", "lines",
    "should", "would", "could", "make", "made", "get", "got", "give", "here", "only", "also",
}
UNCERTAIN: list[str] = []
MAX_FILES = 5000
MAX_FILE_BYTES = 400_000
MAX_SYMBOLS = 40_000


def words(text: str) -> list[str]:
    """Split on anything that is not a letter, digit or underscore, then drop stopwords."""
    out: list[str] = []
    current: list[str] = []
    for character in text:
        if character.isalnum() or character == "_":
            current.append(character)
        else:
            if current:
                out.append("".join(current))
                current = []
    if current:
        out.append("".join(current))
    result: list[str] = []
    for raw in out:
        for part in raw.split("_"):
            part = part.lower()
            if len(part) >= 3 and part not in STOP:
                result.append(part)
    return result


def path_tokens(path: str) -> list[str]:
    return words(path.replace("/", " ").replace(".", " ").replace("-", " "))


def issue_identifiers(issue: str) -> list[str]:
    """Names the issue spells out: CamelCase, snake_case, or anything inside backticks."""
    found: list[str] = []
    for token in words_keep_case(issue):
        stripped = token.strip("`'\"(),.:;")
        looks_camel = any(character.isupper() for character in stripped[1:])
        looks_snake = "_" in stripped
        if len(stripped) >= 4 and (looks_camel or looks_snake):
            found.append(stripped)
    for chunk in quoted_chunks(issue):
        for token in words_keep_case(chunk):
            if len(token) >= 4:
                found.append(token)
    unique: list[str] = []
    for token in found:
        if token not in unique:
            unique.append(token)
    return unique[:20]


def words_keep_case(text: str) -> list[str]:
    out: list[str] = []
    current: list[str] = []
    for character in text:
        if character.isalnum() or character == "_":
            current.append(character)
        else:
            if current:
                out.append("".join(current))
                current = []
    if current:
        out.append("".join(current))
    return out


def quoted_chunks(text: str) -> list[str]:
    """Backticked spans first (most reliable), then double and single quoted spans."""
    chunks: list[str] = []
    for delimiter in ("`", '"', "'"):
        parts = text.split(delimiter)
        if len(parts) >= 3:
            chunks.extend(parts[1::2])
    return [chunk.strip() for chunk in chunks if 2 <= len(chunk.strip()) <= 80]


def issue_keywords(issue: str) -> list[str]:
    counts = Counter(words(issue))
    return [word for word, _ in counts.most_common(30)]


def mentioned_files(issue: str, paths: list[str]) -> dict[str, float]:
    """Files the issue names directly: a path, a stem like console.py, or a dotted module."""
    lowered = issue.lower()
    hits: dict[str, float] = {}
    for path in paths:
        low = path.lower()
        stem = low.rsplit("/", 1)[-1].rsplit(".", 1)[0]
        dotted = low[:-3].replace("/", ".") if low.endswith(".py") else low.replace("/", ".")
        if low in lowered:
            hits[path] = 3.0
        elif stem and (stem + ".py") in lowered:
            hits[path] = 2.0
        elif len(stem) >= 4 and dotted in lowered:
            hits[path] = 2.5
    return hits


def resolve_root(explicit: str | None) -> Path:
    """Find the repository root.

    The ADK skill runner executes this script with the current directory set to its own temporary
    directory, and the subprocess sandbox keeps the repository at <sandbox>/workspace rather than
    /workspace. So the root is detected instead of assumed: an explicit --root wins, then environment
    hints, then the usual locations, and the first candidate that actually contains Python files is used.
    """
    candidates: list[Path] = []
    if explicit:
        candidates.append(Path(explicit))
    for name in ("SWEGEMMA_WORKSPACE", "WORKSPACE", "PWD", "OLDPWD"):
        value = os.environ.get(name)
        if value:
            candidates.append(Path(value))
    here = Path.cwd()
    candidates.extend([Path("/workspace"), here, here.parent])
    # The runner materialises the skill in a temporary directory next to the sandbox workspace, so the
    # repository is often a sibling: look for a "workspace" directory on the way up as well.
    for parent in list(here.parents)[:4]:
        candidates.append(parent / "workspace")
        candidates.append(parent)

    def py_count(candidate: Path, limit: int = 20) -> int:
        count = 0
        try:
            for _ in candidate.rglob("*.py"):
                count += 1
                if count >= limit:
                    break
        except OSError:
            return 0
        return count

    seen: set[str] = set()
    ordered: list[Path] = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            ordered.append(candidate)
    # A real repository has many Python files; the skill's own temporary directory has one or two.
    for minimum in (20, 1):
        for candidate in ordered:
            try:
                if candidate.is_dir() and py_count(candidate, minimum) >= minimum:
                    if minimum < 20:
                        UNCERTAIN.append(str(candidate))
                    return candidate
            except OSError:
                continue
    return ordered[0] if ordered else Path("/workspace")


def collect(root: Path, include_tests: bool) -> dict[str, str]:
    files: dict[str, str] = {}
    try:
        if not root.is_dir():
            return files
        entries = sorted(root.rglob("*.py"))
    except OSError:
        return files
    for path in entries:
        if len(files) >= MAX_FILES:
            break
        parts = {part.lower() for part in path.parts}
        if ".git" in parts or "__pycache__" in parts or "node_modules" in parts or ".venv" in parts:
            continue
        if not include_tests and ("tests" in parts or "testing" in parts or path.name.startswith("test_")):
            continue
        try:
            data = path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            continue
        files[str(path.relative_to(root)).replace("\\", "/")] = data[:MAX_FILE_BYTES]
    return files


def symbol_table(files: dict[str, str]) -> list[tuple[str, str, int, int, list[str]]]:
    """(path, symbol name, start line, end line, body tokens) for every function and class."""
    table: list[tuple[str, str, int, int, list[str]]] = []
    for path, source in files.items():
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in ast.walk(tree):
            if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                continue
            name = getattr(node, "name", "")
            if not name:
                continue
            body = ast.get_source_segment(source, node) or name
            table.append((path, name, node.lineno, getattr(node, "end_lineno", node.lineno), words(body)))
            if len(table) >= MAX_SYMBOLS:
                return table
    return table


def bm25_rank(docs: dict[str, list[str]], query: list[str], k1: float = 1.4, b: float = 0.75) -> dict[str, float]:
    if not docs or not query:
        return {}
    lengths = {name: max(1, len(tokens)) for name, tokens in docs.items()}
    average = sum(lengths.values()) / len(lengths)
    document_frequency = Counter()
    for tokens in docs.values():
        document_frequency.update(set(tokens))
    count = len(docs)
    scores: dict[str, float] = {}
    for name, tokens in docs.items():
        frequencies = Counter(tokens)
        score = 0.0
        for term in query:
            if term not in frequencies:
                continue
            key = term if term in document_frequency else term.lower()
            idf = math.log(1 + (count - document_frequency[key] + 0.5) / (document_frequency[key] + 0.5))
            denominator = frequencies[term] + k1 * (1 - b + b * lengths[name] / average)
            score += idf * frequencies[term] * (k1 + 1) / denominator
        if score > 0:
            scores[name] = score
    return scores


def fuse(rankings: list[dict[str, float]], weights: list[float], k: int = 60) -> dict[str, float]:
    fused: dict[str, float] = {}
    for ranking, weight in zip(rankings, weights):
        for position, (name, _) in enumerate(sorted(ranking.items(), key=lambda pair: -pair[1]), start=1):
            fused[name] = fused.get(name, 0.0) + weight / (k + position)
    return fused


def evidence_for(source: str, terms: list[str], limit: int = 3) -> list[str]:
    hits: list[str] = []
    if not terms:
        return hits
    lowered = [term.lower() for term in terms]
    for number, line in enumerate(source.splitlines(), start=1):
        text = line.lower()
        if any(term in text for term in lowered):
            hits.append(f"{number}: {line.strip()[:160]}")
            if len(hits) >= limit:
                break
    return hits


def rank(issue: str, root: Path, top: int = 10, include_tests: bool = False) -> dict:
    files = collect(root, include_tests)
    if not files:
        return {"query": issue, "root": str(root), "candidates": []}
    identifiers = issue_identifiers(issue)
    keywords = issue_keywords(issue)
    table = symbol_table(files)

    definition_docs: dict[str, list[str]] = {path: [] for path in files}
    symbol_docs: dict[str, list[str]] = {}
    symbol_owner: dict[str, tuple[str, int, int]] = {}
    for path, name, start, end, body in table:
        definition_docs[path].extend(words(name))
        key = f"{path}::{name}::{start}"
        symbol_docs[key] = body
        symbol_owner[key] = (path, start, end)

    code_docs = {path: words(source) for path, source in files.items()}
    literal_docs = {path: [chunk.lower() for chunk in quoted_chunks(source)] for path, source in files.items()}

    symbol_scores = bm25_rank(symbol_docs, keywords)
    best_symbol: dict[str, tuple[float, str, int, int]] = {}
    for key, score in symbol_scores.items():
        path, start, end = symbol_owner[key]
        current = best_symbol.get(path)
        if current is None or score > current[0]:
            best_symbol[path] = (score, key.split("::")[1], start, end)
    symbol_view = {path: value[0] for path, value in best_symbol.items()}

    directory_docs = {
        path: words(" ".join(path.split("/")[:-1])) + words(path.rsplit("/", 1)[-1].rsplit(".", 1)[0])
        for path in files
    }

    rankings = [
        bm25_rank({path: path_tokens(path) for path in files}, keywords),
        bm25_rank(definition_docs, identifiers),
        bm25_rank(code_docs, keywords),
        symbol_view,
        bm25_rank(literal_docs, identifiers + keywords),
        bm25_rank(directory_docs, keywords),
    ]
    weights = [1.2, 1.6, 1.0, 1.3, 0.8, 0.6]
    # Fusion tuned on all 129 public tasks (experiments/fusion_search.py, view_cache.json):
    #   RRF k=10 gives hit@1 .442 / hit@5 .798 / hit@10 .830
    #   the old RRF k=60 gave            .349 /      .698 /       .814
    # Adding 2x the best single-view contribution is what moves it: a file that is first in one view
    # must not lose to a file that is mediocre in six. Do not raise k back to 60.
    fused: dict[str, float] = {}
    best_view: dict[str, float] = {}
    for ranking, weight in zip(rankings, weights):
        for position, (name, _) in enumerate(sorted(ranking.items(), key=lambda pair: -pair[1]), start=1):
            contribution = weight / (10 + position)
            fused[name] = fused.get(name, 0.0) + contribution
            if contribution > best_view.get(name, 0.0):
                best_view[name] = contribution
    for name, value in best_view.items():
        fused[name] = fused.get(name, 0.0) + 2.0 * value
    for path, boost in mentioned_files(issue, list(files)).items():
        fused[path] = fused.get(path, 0.0) + 0.05 * boost
    for path in files:
        if path not in fused:
            fused[path] = 0.0

    ordered = sorted(fused.items(), key=lambda pair: -pair[1])[: max(1, top)]
    candidates = []
    for path, score in ordered:
        symbol, start, end = "", 0, 0
        if path in best_symbol:
            _, symbol, start, end = best_symbol[path]
        candidates.append(
            {
                "path": path,
                "score": round(score, 5),
                "symbol": symbol,
                "lines": f"{start}-{end}" if start else "",
                "evidence": evidence_for(files[path], identifiers[:6] or keywords[:6]),
            }
        )
    return {"query": issue, "root": str(root), "candidates": candidates}


def _interpreter() -> str:
    """The interpreter this script is running under, which is the one that actually has Python files."""
    return sys.executable or "python3"


def run(argv: list[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description="Rank repository files and symbols for an issue.")
    parser.add_argument("issue", nargs="?", default=None)
    parser.add_argument("--root", default=None)
    parser.add_argument("--top", type=int, default=10)
    parser.add_argument("--include-tests", action="store_true")
    parser.add_argument("--stdin", action="store_true")
    args = parser.parse_args(argv)
    issue = sys.stdin.read() if args.stdin or args.issue is None else args.issue
    root = resolve_root(args.root)
    result = rank(issue, root, top=args.top, include_tests=args.include_tests)
    if not result.get("candidates"):
        result["error"] = "no candidates: the repository root was not found or holds no Python files"
        result["cwd"] = os.getcwd()
        result["hint"] = "pass the repository root explicitly, for example --root /workspace"
    elif UNCERTAIN:
        result["warning"] = "the repository root was uncertain; results come from " + UNCERTAIN[-1]
        result["hint"] = "pass --root with the repository directory if these candidates look wrong"
    return result


def main(argv: list[str] | None = None) -> int:
    """Always print JSON: a traceback tells the agent nothing it can act on."""
    try:
        payload = run(argv)
    except Exception as error:  # noqa: BLE001 - the agent must receive something usable
        payload = {
            "error": type(error).__name__,
            "message": str(error)[:400],
            "cwd": os.getcwd(),
            "root": str(resolve_root(None)),
            "interpreter": _interpreter(),
        }
    print(json.dumps(payload, indent=1))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
''',
}

for rel, text in FILES.items():
    target = AGENT_DIR / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    before = target.read_text(encoding="utf-8") if target.is_file() else ""
    target.write_text(text, encoding="utf-8")
    print(f"{rel}: {len(before)} -> {len(text)} chars")

for dead in ("prompts/system.md", "sub_agents/code_analyzer.yaml"):
    target = AGENT_DIR / dead
    if target.is_file():
        target.unlink()
        print("removed", dead)

locate_md = (AGENT_DIR / "prompts/locate.md").read_text(encoding="utf-8")
locate_yaml = (AGENT_DIR / "sub_agents/locate.yaml").read_text(encoding="utf-8")
verify_yaml = (AGENT_DIR / "sub_agents/verify.yaml").read_text(encoding="utf-8")
locate_md = (AGENT_DIR / "prompts/locate.md").read_text(encoding="utf-8")
verify_md = (AGENT_DIR / "prompts/verify.md").read_text(encoding="utf-8")
edit_md = (AGENT_DIR / "prompts/edit.md").read_text(encoding="utf-8")
locate_yaml = (AGENT_DIR / "sub_agents/locate.yaml").read_text(encoding="utf-8")
verify_yaml = (AGENT_DIR / "sub_agents/verify.yaml").read_text(encoding="utf-8")
eval_yaml = (AGENT_DIR / "eval_config.yaml").read_text(encoding="utf-8")
checks = {
    "locate deadline": "chosen the file by call 8" in locate_md,
    "no get_status in locate": "get_status" not in locate_yaml,
    "repo-scout wired": "skills/repo-scout" in locate_yaml,
    "skill files present": (AGENT_DIR / "skills/repo-scout/SKILL.md").is_file() and (AGENT_DIR / "skills/repo-scout/scripts/locate.py").is_file(),
    "verify test-file ban": "Never create or modify a test file" in verify_md,
    "verify keeps get_status": "get_status" in verify_yaml,
    "edit knows the tree is graded": "grades the working tree as it stands" in edit_md,
    "budget 4.5 min": "max_time_minutes: 4.5" in eval_yaml,
    "dead files gone": not (AGENT_DIR / "prompts/system.md").exists() and not (AGENT_DIR / "sub_agents/code_analyzer.yaml").exists(),
}
print(checks)
assert all(checks.values()), checks
print("shipped bundle installed in", AGENT_DIR)


Cell 44f — attach the adapter to the locate agent 

In [ ]:
# # CELL 44F - install the trained LoRA and attach it to the locate agent.
# # Run in the EVAL notebook after 44e, with the adapter attached as a dataset (gemma4-locate-lora).
# import re
# import shutil
# from pathlib import Path

# AGENT_DIR = Path('/kaggle/working/submission')

# found = [path.parent for path in Path('/kaggle/input').rglob('adapter_config.json')]
# assert found, 'attach the dataset that holds adapter_config.json + adapter_model.safetensors'
# source = found[0]
# print('adapter source:', source)
# print('files present:', sorted(p.name for p in source.iterdir() if p.is_file()))

# target = AGENT_DIR / 'adapters' / 'locate_lora'
# target.mkdir(parents=True, exist_ok=True)
# for name in ('adapter_config.json', 'adapter_model.safetensors'):
#     assert (source / name).is_file(), f'{name} is missing from the dataset'
#     shutil.copy2(source / name, target / name)

# locate_yaml = AGENT_DIR / 'sub_agents' / 'locate.yaml'
# text = locate_yaml.read_text(encoding='utf-8')
# if 'adapter:' not in text:
#     text = re.sub(r'(?m)^(model: .*)$', r'\1\nadapter: locate_lora', text, count=1)
#     locate_yaml.write_text(text, encoding='utf-8')
# print(text)

# size = sum(p.stat().st_size for p in target.iterdir() if p.is_file())
# checks = {
#     'adapter files installed': (target / 'adapter_config.json').is_file() and (target / 'adapter_model.safetensors').is_file(),
#     'locate sets the adapter': 'adapter: locate_lora' in locate_yaml.read_text(encoding='utf-8'),
#     'locate keeps the skill': 'skills/repo-scout' in locate_yaml.read_text(encoding='utf-8'),
#     'locate keeps its tools': 'run_command' in locate_yaml.read_text(encoding='utf-8'),
#     'under one GiB': size < 1024 ** 3,
# }
# print(checks)
# assert all(checks.values()), checks
# print(f'installed {size / 2 ** 20:.1f} MiB into {target}')
# print('next: Cell 46c (PICK = S) then Cell 67 on the six tasks')

Cell 45 — use the zip you built, and drop thinking_level for this notebook only

In [ ]:
# from pathlib import Path

# MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2")
# AGENT_DIR = Path("/kaggle/working/submission")
# assert MODEL_PATH.is_dir(), "Attach google/gemma-4 gemma-4-31b-it-qat-w4a16-ct version 2 and restart."
# assert (AGENT_DIR / "agent.yaml").is_file(), "Re-run Cells 40-42. submission/ is missing."

# # Host notebook: LiteLLM maps thinking_level onto reasoning_effort. Omit it here.
# # This does not change the zip in gemma4-dev.
# (AGENT_DIR / "configs" / "sampling.yaml").write_text(
#     "temperature: 0.2\n"
#     "top_p: 0.95\n"
#     "max_output_tokens: 8192\n"
#     "thinking_config:\n"
#     "  thinking_budget: 2048\n"
#     "  include_thoughts: true\n",
#     encoding="utf-8",
# )
# print("model", MODEL_PATH)
# print("agent", AGENT_DIR)
# print("thinking_level", "thinking_level" in (AGENT_DIR / "configs" / "sampling.yaml").read_text())
# Cell 45 - re-declares the paths Cell 46 needs and REPAIRS sampling.yaml when a stale notebook cell
# has stripped thinking_level or top_k. The shipped content is embedded, so this cell is safe to re-run.
# Background: the pre-2026-10-10 Cell 45 rewrote sampling.yaml from scratch and silently dropped both keys.
from pathlib import Path

AGENT_DIR = Path("/kaggle/working/submission")
MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2")

SHIPPED = (
    "temperature: 0.2\n"
    "top_p: 0.95\n"
    "top_k: 64\n"
    "max_output_tokens: 8192\n"
    "thinking_config:\n"
    "  thinking_level: medium\n"
    "  thinking_budget: 2048\n"
    "  include_thoughts: true\n"
)

assert MODEL_PATH.is_dir(), "Attach google/gemma-4 gemma-4-31b-it-qat-w4a16-ct version 2 and restart."
assert (AGENT_DIR / "agent.yaml").is_file(), "Run Cell 44c or Cell 44e first: submission/ is missing."

sampling_path = AGENT_DIR / "configs" / "sampling.yaml"
sampling_path.parent.mkdir(parents=True, exist_ok=True)
sampling = sampling_path.read_text(encoding="utf-8") if sampling_path.is_file() else ""
if "thinking_level" not in sampling or "top_k" not in sampling:
    print("repairing sampling.yaml: a stale cell had stripped thinking_level or top_k")
    sampling_path.write_text(SHIPPED, encoding="utf-8")
    sampling = SHIPPED

print("model", MODEL_PATH)
print("agent", AGENT_DIR)
print("thinking_level present:", "thinking_level" in sampling)
print("top_k present:", "top_k" in sampling)
assert "thinking_level" in sampling and "top_k" in sampling, "sampling.yaml is still not the shipped one"
print(sampling)

Cell 46 — start the server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True
TARGET_MODEL_NAME = "gemma-4-31b-it-qat-w4a16-ct"
INFERENCE_API_KEY = "EMPTY"

# swegemma 0.2.11 (wheelhouse update 2026-10-09 14:43 UTC) removed swegemma.models.discovery.
# The declared-model walk now lives in adk_submission.discovery; swegemma.cli imports it from there.
def _normalize(name: str) -> str:
    for prefix in ("openai/", "google/", "hosted_vllm/", "custom/"):
        if name.startswith(prefix):
            return name[len(prefix):]
    return name


declared_models = sorted({_normalize(model) for model in discover_declared_models(AGENT_DIR)})
print("declared_models", declared_models)
assert declared_models == [TARGET_MODEL_NAME], f"single-model rule: {declared_models}"

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
print("adapters", adapters)

gpu_count = torch.cuda.device_count()
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host="127.0.0.1",
    tool_call_parser="gemma4",
    reasoning_parser="gemma4",
    default_chat_template_kwargs={"enable_thinking": True},
    max_model_len=32768,
    dtype="bfloat16" if torch.cuda.is_bf16_supported() else "auto",
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f"vLLM server started on {server_instance.base_url} (tp={tp_size})")
models = server_instance.create_model_registry(
    aliases=[TARGET_MODEL_NAME],
    model_prefix="openai/",
    api_key=INFERENCE_API_KEY,
)

Cell 46b

In [ ]:
print("server_up", server_instance.base_url)
print("registry_type", type(models).__name__)

Cell 46c - local sweep budget (15 min / 50 calls)

In [ ]:
# Cell 46c - write the SHIPPED budget, print it, and prove it parses the way Cell 67 reads it.
# Two things this cell must get right, both learned the hard way:
#   * the file keeps its top-level "evaluation:" mapping, because Cell 67 does
#     yaml.safe_load(eval_config.yaml)["evaluation"];
#   * ab_budget.json is rewritten too, otherwise Cell 67 prints a stale "budget in force" line.
import json
from pathlib import Path

import yaml

AGENT_DIR = Path("/kaggle/working/submission")
EVAL = AGENT_DIR / "eval_config.yaml"
BUDGET_LOG = Path("/kaggle/working/ab_budget.json")

PICK = "S"
SETTINGS = {
    "A": {"timeout_seconds": 120, "max_tool_calls": 20, "max_time_minutes": 8.0, "max_turns": 30},
    "B": {"timeout_seconds": 120, "max_tool_calls": 50, "max_time_minutes": 15.0, "max_turns": 80},
    "S": {"timeout_seconds": 300, "max_tool_calls": 30, "max_time_minutes": 4.5, "max_turns": 60},
}
assert PICK in SETTINGS, PICK
chosen = SETTINGS[PICK]
assert chosen["max_time_minutes"] == 4.5, "the shipped bundle declares 4.5 minutes"
assert chosen["max_time_minutes"] * 120 + 60 <= 720, "120 tasks must fit the 12-hour submission cap"

EVAL.write_text(
    "evaluation:\n"
    "  timeout_seconds: {timeout_seconds}\n"
    "  max_tool_calls: {max_tool_calls}\n"
    "  max_time_minutes: {max_time_minutes}\n"
    "  max_turns: {max_turns}\n".format(**chosen),
    encoding="utf-8",
)
BUDGET_LOG.write_text(json.dumps({"pick": PICK, **chosen}, indent=1), encoding="utf-8")

print(EVAL.read_text(encoding="utf-8"))
print("ab_budget.json:", BUDGET_LOG.read_text(encoding="utf-8"))

# the exact read Cell 67 performs
parsed = yaml.safe_load(EVAL.read_text(encoding="utf-8"))
assert "evaluation" in parsed, "eval_config.yaml lost its 'evaluation:' wrapper"
assert parsed["evaluation"]["max_time_minutes"] == 4.5
assert parsed["evaluation"]["max_tool_calls"] == 30
assert parsed["evaluation"]["timeout_seconds"] == 300
assert parsed["evaluation"]["max_turns"] == 60
print("parses as Cell 67 expects | budget", PICK, "| 4.5 min / 30 calls | 12h check ok")

Cell 67 - task run (swap TASK_ID)

In [ ]:
# Cell 67 - panel runner. Set TASK_IDS and run once: every task is evaluated, saved and summarised.
import asyncio
import concurrent.futures
import json
from pathlib import Path

import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks
from swegemma.results import append_task_result

assert "models" in dir(), "Server state is gone: re-run Cell 46 before this cell."
AGENT_DIR = Path("/kaggle/working/submission")

# TASK_IDS = ["fastapi_14349", "fastapi_14356", "fastapi_14361", "fastapi_14430", "fastapi_14458", "fastapi_14479", "fastapi_14873", "fastapi_15023", "fastapi_15588", "fastapi_9425", "requests_7205", "requests_7328", "requests_7427", "requests_7502", "rich_2725", "rich_3470", "rich_3521", "rich_3782", "rich_3882", "rich_4079"]# rich_3882 first = canary
TASK_IDS = ["rich_3882"]
WORKING = Path("/kaggle/working")
BUDGET_LOG = WORKING / "ab_budget.json"

VERDICT = {
    0: "RESOLVED",
    1: "test failed (agent)",
    2: "collection error (missing deps - not the agent)",
    -1: "budget death (agent, no patch)",
}


def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


budget = json.loads(BUDGET_LOG.read_text(encoding="utf-8")) if BUDGET_LOG.is_file() else {}
print("budget in force:", budget or "unlogged")

DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
TASKS = load_tasks(DATA_DIR / "tasks.jsonl")
ev = yaml.safe_load((AGENT_DIR / "eval_config.yaml").read_text(encoding="utf-8"))["evaluation"]
limits, gen_constraints = build_submission_limits()

rows = []
for index, task_id in enumerate(TASK_IDS, start=1):
    task = next((t for t in TASKS if t.instance_id == task_id), None)
    if task is None:
        print(f"[{index}/{len(TASK_IDS)}] {task_id}: not in the public task list, skipped")
        continue
    results_dir = WORKING / f"results_{task_id}"
    eval_config = EvalConfig(
        tasks_path=DATA_DIR / "tasks.jsonl",
        snapshots_dir=DATA_DIR / "snapshots",
        results_dir=results_dir,
        submission_dir=AGENT_DIR,
        models=models,
        sandbox="subprocess",
        timeout_seconds=int(ev["timeout_seconds"]),
        max_time_minutes=float(ev["max_time_minutes"]),
        max_tool_calls=int(ev["max_tool_calls"]),
        max_turns=int(ev["max_turns"]),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15,
            overlap_size=2,
            token_threshold=14336,
            event_retention_size=5,
        ),
        graph_dir=str(DATA_DIR / "graphs"),
        embeddings_dir=str(DATA_DIR / "embeddings"),
        wheels_dir=DATA_DIR / "wheels",
        verbose=False,
    )
    result = run_sync(Evaluator(eval_config).evaluate_task, task=task, task_index=index, total_tasks=len(TASK_IDS))
    append_task_result(result, results_dir)
    patch = result.agent_patch or ""
    code = result.test_exit_code
    files = [line.split("diff --git a/")[-1] for line in patch.splitlines() if line.startswith("diff --git ")]
    rows.append(
        {
            "task": task_id,
            "exit": code,
            "resolved": bool(result.resolved),
            "patch_chars": len(patch),
            "tool_calls": result.tool_calls,
            "duration_s": round(result.duration_seconds, 1),
            "files": files,
        }
    )
    print(
        f"[{index}/{len(TASK_IDS)}] instance={task_id} exit={code} [{VERDICT.get(code, '?')}] "
        f"resolved={result.resolved} patch_chars={len(patch)} tool_calls={result.tool_calls} "
        f"duration={result.duration_seconds:.1f}s files={files}"
    )

print()
print("| task | exit | resolved | patch_chars | tool_calls | duration_s |")
print("|---|---|---|---|---|---|")
for row in rows:
    print(
        f"| {row['task']} | {row['exit']} | {row['resolved']} | {row['patch_chars']} | "
        f"{row['tool_calls']} | {row['duration_s']} |"
    )
resolved = sum(1 for row in rows if row["resolved"])
print(f"resolved {resolved}/{len(rows)}   budget {budget.get('pick', '?')} "
      f"({ev['max_time_minutes']} min / {ev['max_tool_calls']} calls)")
(WORKING / "panel_last.json").write_text(json.dumps({"budget": budget, "rows": rows}, indent=2), encoding="utf-8")
print("saved", WORKING / "panel_last.json")

Purge

In [ ]:
# PURGE CELL - free the GPUs before training.
# The evaluation notebook's vLLM server holds about 20 GiB on every L4. This kills only processes
# that nvidia-smi reports as holding GPU memory, never this kernel, then reports what is free.
import os
import signal
import subprocess
import time

import torch

SELF = {os.getpid(), os.getppid()}
listing = subprocess.run(
    ['nvidia-smi', '--query-compute-apps=pid,process_name,used_memory', '--format=csv'],
    capture_output=True, text=True).stdout
print(listing)

killed = []
for line in listing.splitlines()[1:]:
    parts = [part.strip() for part in line.split(',')]
    if len(parts) < 2 or not parts[0].isdigit():
        continue
    pid = int(parts[0])
    if pid in SELF:
        print('keeping this kernel:', pid)
        continue
    try:
        os.kill(pid, signal.SIGTERM)
        killed.append(pid)
    except Exception as error:
        print('SIGTERM failed for', pid, error)

time.sleep(5)
for pid in killed:
    try:
        os.kill(pid, signal.SIGKILL)
    except Exception:
        pass
time.sleep(3)
torch.cuda.empty_cache()

free = [torch.cuda.mem_get_info(index)[0] / 2 ** 30 for index in range(torch.cuda.device_count())]
print('killed:', killed)
print([f'gpu{index}: {value:.1f} GiB free' for index, value in enumerate(free)])
if min(free) < 8:
    print('Still occupied. The memory belongs to something outside this kernel:')
    print('use Runtime -> Restart session, then run the training cell as the first and only cell.')
else:
    print('GPUs are clear: now run the training cell.')

Cell 68 TRAIN CELL - LoRA

In [ ]:
# TRAIN CELL - LoRA for the locate role. Run this in a NEW notebook on L4x4.
# Attach: gemma4-train-wheels (peft/accelerate), the google/gemma-4 model instance
# gemma-4-31b-it-qat-q4_0-unquantized, and the gemma-4-developer-agent competition data.
# Internet must stay OFF. Nothing here touches the scoring pipeline; the adapter it writes is the deliverable.
import glob
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import os as _os
_os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

WHEELS = '/kaggle/input/gemma4-train-wheels'
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELS,
                      '--no-deps', 'peft', 'accelerate'], capture_output=True, text=True).stdout[-600:])

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

TASKS = '/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl'
WORK = Path('/kaggle/working')
OUT_ADAPTER = WORK / 'locate_lora'
STEPS = int(os.environ.get('LOCATE_STEPS', '60'))
HELD_OUT = 20

# ---- mine the training records (issue -> file / symbol / lines) ----
FILE_LINE = re.compile(r'^(?:\+\+\+ b/|--- a/)(\S+)')
HUNK = re.compile(r'^@@ -\d+(?:,\d+)? \+(\d+)(?:,(\d+))? @@ ?(.*)$')

def parse_targets(patch):
    targets, current = [], None
    for line in (patch or '').splitlines():
        match = FILE_LINE.match(line)
        if match:
            path = match.group(1)
            if path == '/dev/null':
                continue
            if current is None or current['file'] != path:
                current = {'file': path, 'ranges': [], 'symbol': ''}
                targets.append(current)
            continue
        if current is None:
            continue
        hunk = HUNK.match(line)
        if hunk:
            start = int(hunk.group(1))
            length = int(hunk.group(2) or '1')
            if length > 0:
                current['ranges'].append([start, start + length - 1])
            context = (hunk.group(3) or '').strip()
            if context and not current['symbol'] and re.match(r'^(def|class|async def)\b', context):
                current['symbol'] = context.split('(')[0].replace('def ', '').replace('class ', '').strip()
    return targets

def answer_text(targets):
    parts = []
    for target in targets:
        spans = ', '.join(f'{a}-{b}' if b > a else str(a) for a, b in target['ranges'])
        symbol = f' {target["symbol"]}' if target['symbol'] else ''
        parts.append(f'{target["file"]}{symbol} lines {spans}')
    return '; '.join(parts)

records = []
for line in Path(TASKS).read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    task = json.loads(line)
    targets = parse_targets(task.get('patch', ''))
    if targets and task.get('problem_statement'):
        records.append({'instance_id': task['instance_id'], 'issue': task['problem_statement'][:3000],
                        'targets': targets, 'answer': answer_text(targets)})
print('records:', len(records))

SYSTEM = ('You localise a software issue in a Python repository. Reply with one line per file, '
          'exactly this shape: path/to/file.py symbol_name lines START-END. No prose, no code.')
def prompt(record):
    return f'Issue:\n{record["issue"]}\n\nWhere is the fix?'

def to_texts(record, tokenizer):
    chat = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt(record)}]
    head = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    return head, head + record['answer'] + tokenizer.eos_token

# ---- find an UNQUANTIZED training base -----------------------------
import json as _json


def pick_base_model(root=Path('/kaggle/input/models')):
    """Prefer the unquantized QAT sibling. Never pick a compressed/quantized serving model: loading
    one drags in compressed_tensors and torch.compile, which is not what training wants."""
    scored = []
    if not Path(root).exists():
        return scored
    for config in Path(root).rglob('config.json'):
        path = config.parent
        text = str(path).lower()
        if 'w4a16' in text or '/other/' in text or 'qat-w4a16' in text:
            continue
        try:
            payload = _json.loads(config.read_text(encoding='utf-8'))
        except Exception:
            continue
        if payload.get('quantization_config'):
            continue
        score = 0
        if 'qat-q4_0-unquantized' in text:
            score += 3
        if 'gemma-4-31b-it' in text:
            score += 2
        if any(path.glob('*.safetensors')):
            score += 1
        scored.append((score, str(path)))
    scored.sort(key=lambda pair: (-pair[0], len(pair[1])))
    return scored


ranked = pick_base_model()
if not ranked or ranked[0][0] < 2:
    raise SystemExit('Attach a google/gemma-4 model instance to this notebook (Add Input -> Models): '
                     'gemma-4-31b-it-qat-q4_0-unquantized is preferred, gemma-4-31b-it also works. '
                     'The competition model gemma-4-31b-it-qat-w4a16-ct cannot be trained on.')
MODEL_DIR = Path(ranked[0][1])
print('base model:', MODEL_DIR)
print('candidates seen:', [p for _, p in ranked[:4]])

# ---- split and tokenise -------------------------------------------
records = records[::1]
held = records[:HELD_OUT]
train = records[HELD_OUT:]
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class Dataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = [to_texts(r, tokenizer) for r in rows]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        prompt_text, full_text = self.rows[index]
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
        ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=3072)['input_ids']
        labels = list(ids)
        for position in range(min(len(prompt_ids), len(labels))):
            labels[position] = -100
        return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(labels)}

print(subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
free_gib = [torch.cuda.mem_get_info(index)[0] / 2**30 for index in range(torch.cuda.device_count())]
print([f'gpu{index}: {value:.1f} GiB free' for index, value in enumerate(free_gib)])
if min(free_gib) < 8:
    raise SystemExit(
        'The GPUs are already occupied: ' + ', '.join(f'gpu{i} {v:.1f} GiB free' for i, v in enumerate(free_gib))
        + '. This is normally the vLLM inference server from the evaluation notebook (about 20 GiB per GPU).'
        + ' Training needs its own session: Runtime -> Restart session and run ONLY this cell,'
        + ' or open a separate notebook that does not start the server.')
torch.cuda.empty_cache()
LOAD_4BIT = _os.environ.get('LOCATE_LOAD_4BIT', '1') == '1'
max_memory = {index: '20GiB' for index in range(torch.cuda.device_count())} or None
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), quantization_config=quant,
                                               device_map='auto', max_memory=max_memory, dtype=torch.bfloat16)
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), dtype=torch.bfloat16,
                                               device_map='auto', max_memory=max_memory)
model.config.use_cache = False
print('loaded in 4-bit:', LOAD_4BIT)
print([f'gpu{i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB' for i in range(torch.cuda.device_count())])
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj.linear', 'k_proj.linear', 'v_proj.linear', 'o_proj.linear'])
# Gemma 4 wraps every projection in Gemma4ClippableLinear, so the LoRA attaches to the inner
# .linear child (a Linear4bit after the 4-bit load). Cell 69 strips the .linear suffix from the
# saved keys so the adapter matches the host sample format: ...self_attn.q_proj.lora_A.weight.
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainer_args = {
    'output_dir': str(WORK / 'lora_runner'),
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,
    'learning_rate': 1e-4,
    'max_steps': STEPS,
    'logging_steps': 5,
    'save_steps': STEPS,
    'bf16': True,
    'optim': 'paged_adamw_8bit' if torch.cuda.is_available() else 'adamw_torch',
    'report_to': [],
    'remove_unused_columns': False,
    'gradient_checkpointing': True,
}
from transformers import Trainer, TrainingArguments
trainer = Trainer(model=model, args=TrainingArguments(**trainer_args), train_dataset=Dataset(train))
trainer.train()
model.save_pretrained(str(OUT_ADAPTER))
tokenizer.save_pretrained(str(OUT_ADAPTER))
print('adapter written to', OUT_ADAPTER)
print(sorted(p.name for p in OUT_ADAPTER.iterdir()))

# ---- held-out evaluation: does the model name the right file? ------
model.eval()
def generate(record, max_new_tokens=64):
    chat = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt(record)}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

hits, total = 0, 0
for record in held:
    answer = generate(record)
    gold = {target['file'] for target in record['targets']}
    hit = any(path in answer for path in gold)
    hits += int(hit)
    total += 1
    print(f"{record['instance_id']} gold={sorted(gold)[0]} hit={hit} out={answer.strip()[:70]!r}")
print(f'held-out file hit rate: {hits}/{total} = {hits / max(1, total):.1%}')
print('compare against the retrieval baseline: repo-scout hit@10 = 78.3% on all 129 public tasks')

Cell 69 Package Trained Adapter

In [ ]:
    # # CELL 69 - package the trained adapter so it can become part of a submission.
# # Run in the TRAINING notebook, right after the training cell. Nothing here needs a GPU.
# import json
# import shutil
# import subprocess
# from pathlib import Path

# ADAPTER = Path('/kaggle/working/locate_lora')
# BUNDLE = Path('/kaggle/working/adapters/locate_lora')
# ZIP = Path('/kaggle/working/locate_lora.zip')

# config_path = ADAPTER / 'adapter_config.json'
# weights_path = ADAPTER / 'adapter_model.safetensors'
# assert config_path.is_file() and weights_path.is_file(), 'run the training cell first: ' + str(ADAPTER)

# config = json.loads(config_path.read_text(encoding='utf-8'))
# targets = config.get('target_modules') or []
# print('rank:', config.get('r'), '| alpha:', config.get('lora_alpha'), '| target modules:', len(targets))
# print('base:', config.get('base_model_name_or_path'))
# print('task type:', config.get('task_type'))

# # The harness expects adapters/<name>/adapter_config.json + adapter_model.safetensors, nothing else.
# BUNDLE.mkdir(parents=True, exist_ok=True)
# for path in (config_path, weights_path):
#     shutil.copy2(path, BUNDLE / path.name)

# total = sum(p.stat().st_size for p in BUNDLE.rglob('*') if p.is_file())
# print('bundle files:', [p.name for p in sorted(BUNDLE.iterdir())])
# print(f'unpacked size: {total / 2 ** 20:.1f} MiB (submission limit 3072 MiB)')
# assert total < 3 * 1024 ** 3, 'the adapter alone is over the 3 GiB submission limit'

# subprocess.run(['bash', '-lc', 'cd /kaggle/working && rm -f locate_lora.zip && zip -r locate_lora.zip adapters'],
#                check=False)
# print('download this from the notebook output panel:', ZIP, f'({ZIP.stat().st_size / 2 ** 20:.1f} MiB)')
# print('when uploading it as a dataset keep the folder name: adapters/locate_lora/')

Cell 70 Train re-ranker

In [ ]:
# CELL 70 - train the locate RERANKER: issue + ten candidate paths -> the one to edit.
# Replaces the Cell 68 formulation, which asked the model to recall paths without seeing the repo and
# scored 25% against a 78.3% retrieval baseline. Here the repository's own ranking is in the prompt.
# Run on L4x4 with internet OFF, in a notebook that is NOT running the vLLM server.
import glob
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

WHEELS = '/kaggle/input/gemma4-train-wheels'
if Path(WHEELS).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELS,
                    '--no-deps', 'peft', 'accelerate'], capture_output=True, text=True)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

WORK = Path('/kaggle/working')
OUT = WORK / 'locate_reranker'
STEPS = int(os.environ.get('RERANKER_STEPS', '150'))
HELD_OUT = 20
SERVED_MODEL = 'google/gemma-4-31b-it-qat-w4a16-ct'


def pick_base_model(root=Path('/kaggle/input/models')):
    scored = []
    if not Path(root).exists():
        return scored
    for config in Path(root).rglob('config.json'):
        path = config.parent
        text = str(path).lower()
        if 'w4a16' in text or '/other/' in text:
            continue
        try:
            payload = json.loads(config.read_text(encoding='utf-8'))
        except Exception:
            continue
        if payload.get('quantization_config'):
            continue
        score = 0
        if 'qat-q4_0-unquantized' in text:
            score += 3
        if 'gemma-4-31b-it' in text:
            score += 2
        if any(path.glob('*.safetensors')):
            score += 1
        scored.append((score, str(path)))
    scored.sort(key=lambda pair: (-pair[0], len(pair[1])))
    return scored


ranked = pick_base_model()
if not ranked or ranked[0][0] < 2:
    raise SystemExit('Attach the gemma-4-31b-it-qat-q4_0-unquantized instance first (Add Input -> Models).')
MODEL_DIR = Path(ranked[0][1])
print('base model:', MODEL_DIR)

free_gib = [torch.cuda.mem_get_info(index)[0] / 2**30 for index in range(torch.cuda.device_count())]
print([f'gpu{index}: {value:.1f} GiB free' for index, value in enumerate(free_gib)])
if min(free_gib) < 8:
    raise SystemExit('GPUs are occupied — usually the vLLM server. Restart the session and run only this cell.')

records_path = None
for candidate in Path('/kaggle/input').rglob('reranker_records.jsonl'):
    records_path = candidate
    break
if records_path is None:
    raise SystemExit('Attach the gemma4-reranker-data dataset (reranker_records.jsonl).')

rows = [json.loads(line) for line in records_path.read_text(encoding='utf-8').splitlines() if line.strip()]
usable = [row for row in rows if row.get('gold_in_list') and row.get('answer') != 'NONE']
print('records:', len(rows), '| usable (gold is in the candidate list):', len(usable))

SYSTEM = ('You choose which file must be edited to fix a software issue. Answer with exactly one path '
          'from the candidate list, and nothing else.')


def prompt_for(row):
    lines = [f'{index + 1}. {path}' for index, path in enumerate(row['candidates'])]
    return (f"Issue:\n{row['issue']}\n\nCandidate files:\n" + '\n'.join(lines)
            + '\n\nWhich file must be edited to fix this issue? Answer with its path only.')


held = usable[:HELD_OUT]
train = usable[HELD_OUT:]
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def encode(row):
    chat = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt_for(row)}]
    head = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    full = head + row['answer'] + tokenizer.eos_token
    prompt_ids = tokenizer(head, add_special_tokens=False)['input_ids']
    ids = tokenizer(full, add_special_tokens=False, truncation=True, max_length=2048)['input_ids']
    labels = list(ids)
    for position in range(min(len(prompt_ids), len(labels))):
        labels[position] = -100
    return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(labels)}


class Rows(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = [encode(row) for row in data]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
max_memory = {index: '20GiB' for index in range(torch.cuda.device_count())} or None
model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), quantization_config=quant,
                                             device_map='auto', max_memory=max_memory, dtype=torch.bfloat16)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj.linear', 'k_proj.linear', 'v_proj.linear', 'o_proj.linear'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir=str(WORK / 'reranker_runner'), per_device_train_batch_size=1,
                           gradient_accumulation_steps=4, learning_rate=2e-4, max_steps=STEPS,
                           logging_steps=5, save_steps=STEPS, bf16=True, optim='paged_adamw_8bit',
                           report_to=[], remove_unused_columns=False, gradient_checkpointing=True),
    train_dataset=Rows(train),
)
trainer.train()
model.save_pretrained(str(OUT))
tokenizer.save_pretrained(str(OUT))
print('adapter written to', OUT, sorted(p.name for p in OUT.iterdir()))

model.eval()


def choose(row, max_new_tokens=40):
    chat = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt_for(row)}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


right = 0
rank1 = 0
for row in held:
    answer = choose(row)
    hit = row['answer'] in answer or any(gold in answer for gold in row['gold'])
    right += int(hit)
    rank1 += int(row['candidates'].index(row['answer']) == 0)
    print(f"{row['instance_id']} gold_rank={row['candidates'].index(row['answer']) + 1} "
          f"hit={hit} out={answer[:70]!r}")
print(f'held-out selection accuracy: {right}/{len(held)} = {right / max(1, len(held)):.1%}')
print(f'the retrieval ranking already scored rank-1 on {rank1}/{len(held)} of the same tasks')
print('acceptance bar: the adapter must beat the retrieval top-1 (38.8% over all 129 tasks)')

cell 71 adapter_vs_base

In [ ]:
# CELL 71 - the decisive comparison: does the adapter beat the BASE MODEL on the same prompt?
# Cell 70 scored 11/20 with the adapter, against 9/20 for the retrieval ranking. But the loss stayed flat,
# so the selection skill may come from the candidate list in the prompt rather than from LoRA. This runs
# both conditions on the same 20 held-out tasks in one pass. If the base model matches the adapter, we
# ship a prompt change instead of an adapter: no artifact, no KV-cache cost, no training.
import glob
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

WHEELS = '/kaggle/input/gemma4-train-wheels'
if Path(WHEELS).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELS,
                    '--no-deps', 'peft', 'accelerate'], capture_output=True, text=True)

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

HELD_OUT = 20
SYSTEM = ('You choose which file must be edited to fix a software issue. Answer with exactly one path '
          'from the candidate list, and nothing else.')


def pick_base_model(root=Path('/kaggle/input/models')):
    scored = []
    for config in Path(root).rglob('config.json'):
        path = config.parent
        text = str(path).lower()
        if 'w4a16' in text or '/other/' in text:
            continue
        try:
            payload = json.loads(config.read_text(encoding='utf-8'))
        except Exception:
            continue
        if payload.get('quantization_config'):
            continue
        score = 0
        if 'qat-q4_0-unquantized' in text:
            score += 3
        if 'gemma-4-31b-it' in text:
            score += 2
        if any(path.glob('*.safetensors')):
            score += 1
        scored.append((score, str(path)))
    scored.sort(key=lambda pair: (-pair[0], len(pair[1])))
    return scored


records_path = None
for candidate in Path('/kaggle/input').rglob('reranker_records.jsonl'):
    records_path = candidate
    break
assert records_path is not None, 'attach the gemma4-reranker-data dataset'
rows = [json.loads(line) for line in records_path.read_text(encoding='utf-8').splitlines() if line.strip()]
usable = [row for row in rows if row.get('gold_in_list') and row.get('answer') != 'NONE']
held = usable[:HELD_OUT]
print('held-out tasks:', [row['instance_id'] for row in held])


def prompt_for(row):
    lines = [f'{index + 1}. {path}' for index, path in enumerate(row['candidates'])]
    return (f"Issue:\n{row['issue']}\n\nCandidate files:\n" + '\n'.join(lines)
            + '\n\nWhich file must be edited to fix this issue? Answer with its path only.')


free_gib = [torch.cuda.mem_get_info(index)[0] / 2**30 for index in range(torch.cuda.device_count())]
print([f'gpu{index}: {value:.1f} GiB free' for index, value in enumerate(free_gib)])
# Use every GPU with room and mark the tight ones unavailable, so an earlier cell in this same session
# does not force a restart. 4-bit weights need roughly 4.5 GiB per shard.
plan = {index: (f'{max(4, int(value) - 2)}GiB' if value >= 8 else '0GiB')
        for index, value in enumerate(free_gib)}
available = sum(int(size[:-3]) for size in plan.values())
print('device plan:', plan, '| usable GiB:', available)
if available < 20:
    raise SystemExit('Not enough free GPU memory (' + str(available) + ' GiB). Restart the session and '
                     'run only this cell, or purge the vLLM server first.')
max_memory = plan

MODEL_DIR = Path(pick_base_model()[0][1])
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
max_memory = {index: '20GiB' for index in range(torch.cuda.device_count())} or None
model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), quantization_config=quant,
                                             device_map='auto', max_memory=max_memory, dtype=torch.bfloat16)
model.eval()


def choose(row, max_new_tokens=40):
    chat = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt_for(row)}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def evaluate(label):
    right = 0
    detail = []
    for row in held:
        answer = choose(row)
        hit = row['answer'] in answer or any(gold in answer for gold in row['gold'])
        right += int(hit)
        detail.append((row['instance_id'], row['candidates'].index(row['answer']) + 1, hit, answer[:60]))
    print(f'--- {label}: {right}/{len(held)} = {right / len(held):.1%}')
    for instance, rank, hit, answer in detail:
        print(f'   {instance:16} gold_rank={rank:<3} hit={hit} out={answer!r}')
    return right, {instance: hit for instance, _, hit, _ in detail}


base_right, base_hits = evaluate('BASE MODEL, same prompt (no adapter)')

adapter_dirs = [Path('/kaggle/working/locate_reranker')]
adapter_dirs += [path.parent.parent for path in Path('/kaggle/input').rglob('adapter_config.json')]
adapter = next((path for path in adapter_dirs if path.is_dir()), None)
if adapter is None:
    print('no adapter found; the base result above is the whole answer')
else:
    print('adapter:', adapter)
    model = PeftModel.from_pretrained(model, str(adapter))
    model.eval()
    adapter_right, adapter_hits = evaluate('WITH ADAPTER (cell 70)')

    agree = sum(1 for instance in base_hits if base_hits[instance] == adapter_hits.get(instance))
    print()
    print(f'base {base_right}/20  vs  adapter {adapter_right}/20')
    print(f'tasks where both agree: {agree}/20')
    print('retrieval rank-1 on these same tasks: '
          f"{sum(1 for row in held if row['candidates'].index(row['answer']) == 0)}/20")
    if adapter_right <= base_right:
        print('VERDICT: the adapter adds nothing on this prompt. Ship the candidate-list prompt instead.')
    elif adapter_right - base_right < 3:
        print('VERDICT: within noise on 20 tasks. Prefer the prompt-only route: no artifact to carry.')
    else:
        print('VERDICT: the adapter earns its place. Package it with Cell 69 and evaluate it in a panel.')


CELL 72 - score the PROMPT-ONLY Eval

In [ ]:
# CELL 72 - score the PROMPT-ONLY selector over every usable task.
# Cell 71 proved the adapter changes nothing (base 11/20 == adapter 11/20, byte-identical outputs), so the
# gain measured on the 20-task hold-out comes from the candidate list in the prompt. This measures that
# prompt over all 107 tasks whose gold file is in the list, and breaks the result down by the rank the
# retrieval already gave the gold file. No training, just generation.
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

WHEELS = '/kaggle/input/gemma4-train-wheels'
if Path(WHEELS).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELS,
                    '--no-deps', 'peft', 'accelerate'], capture_output=True, text=True)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LIMIT = int(os.environ.get('SELECTOR_LIMIT', '0'))  # 0 means every usable record
SYSTEM = ('You choose which file must be edited to fix a software issue. Answer with exactly one path '
          'from the candidate list, and nothing else.')


def pick_base_model(root=Path('/kaggle/input/models')):
    scored = []
    for config in Path(root).rglob('config.json'):
        path = config.parent
        text = str(path).lower()
        if 'w4a16' in text or '/other/' in text:
            continue
        try:
            payload = json.loads(config.read_text(encoding='utf-8'))
        except Exception:
            continue
        if payload.get('quantization_config'):
            continue
        score = 0
        if 'qat-q4_0-unquantized' in text:
            score += 3
        if 'gemma-4-31b-it' in text:
            score += 2
        if any(path.glob('*.safetensors')):
            score += 1
        scored.append((score, str(path)))
    scored.sort(key=lambda pair: (-pair[0], len(pair[1])))
    return scored


records_path = None
for candidate in Path('/kaggle/input').rglob('reranker_records.jsonl'):
    records_path = candidate
    break
assert records_path is not None, 'attach the gemma4-reranker-data dataset'
rows = [json.loads(line) for line in records_path.read_text(encoding='utf-8').splitlines() if line.strip()]
usable = [row for row in rows if row.get('gold_in_list') and row.get('answer') != 'NONE']
if LIMIT:
    usable = usable[:LIMIT]
print('records:', len(rows), '| evaluating:', len(usable))


def prompt_for(row):
    lines = [f'{index + 1}. {path}' for index, path in enumerate(row['candidates'])]
    return (f"Issue:\n{row['issue']}\n\nCandidate files:\n" + '\n'.join(lines)
            + '\n\nWhich file must be edited to fix this issue? Answer with its path only.')


free_gib = [torch.cuda.mem_get_info(index)[0] / 2**30 for index in range(torch.cuda.device_count())]
print([f'gpu{index}: {value:.1f} GiB free' for index, value in enumerate(free_gib)])
plan = {index: (f'{max(4, int(value) - 2)}GiB' if value >= 8 else '0GiB')
        for index, value in enumerate(free_gib)}
available = sum(int(size[:-3]) for size in plan.values())
print('device plan:', plan, '| usable GiB:', available)
if available < 20:
    raise SystemExit('Not enough free GPU memory. Restart the session and run only this cell.')

MODEL_DIR = Path(pick_base_model()[0][1])
print('base model:', MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), quantization_config=quant,
                                             device_map='auto', max_memory=plan, dtype=torch.bfloat16)
model.eval()


def choose(row, max_new_tokens=40):
    chat = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt_for(row)}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


buckets = {'rank1': [0, 0], 'rank2to5': [0, 0], 'rank6to10': [0, 0]}
per_repo = {}
right = 0
for index, row in enumerate(usable, start=1):
    answer = choose(row)
    gold_rank = row['candidates'].index(row['answer']) + 1
    hit = row['answer'] in answer or any(gold in answer for gold in row['gold'])
    right += int(hit)
    key = 'rank1' if gold_rank == 1 else ('rank2to5' if gold_rank <= 5 else 'rank6to10')
    buckets[key][0] += int(hit)
    buckets[key][1] += 1
    repo = (row.get('repo') or '?').split('/')[-1]
    entry = per_repo.setdefault(repo, [0, 0])
    entry[0] += int(hit)
    entry[1] += 1
    print(f"[{index}/{len(usable)}] {row['instance_id']:16} gold_rank={gold_rank:<3} hit={hit} out={answer[:50]!r}", flush=True)

print()
print(f'prompt-only selection accuracy: {right}/{len(usable)} = {right / len(usable):.1%}')
print('the retrieval top-1 (list order) would score: '
      f"{buckets['rank1'][1]}/{len(usable)} = {buckets['rank1'][1] / len(usable):.1%}")
for key, (hit, count) in buckets.items():
    print(f'  gold was at {key:9}: adapter-level hit {hit}/{count} = {hit / max(1, count):.0%}')
print('per repo:')
for repo, (hit, count) in sorted(per_repo.items()):
    print(f'  {repo:9} {hit:3}/{count:3} = {hit / count:.0%}')
print()
print('acceptance bar: the shipped locate prompt already reports the list order; this prompt is only')
print('worth keeping if it beats the rank-1 column above by more than noise.')

Cell 73

In [ ]:
# Cell 73 - WHERE DOES THE PER-TASK TIME GO? No GPU, no server, runs in seconds.
# The fair panel: 12/20 tasks died with no patch at ~301 s. Both caps bind at once (30 calls x ~10 s =
# 300 s), so before changing any prompt we need to know whether the agent never edited, edited late, or
# spent the clock on slow tools. This reads the traces the panel already wrote.
import json
from collections import Counter
from pathlib import Path

ROOT = Path("/kaggle/working")
traces = sorted(ROOT.glob("results_*/traces/trace_*.json"))
panel = {}
if (ROOT / "panel_last.json").is_file():
    raw = json.loads((ROOT / "panel_last.json").read_text(encoding="utf-8"))
    if isinstance(raw, list):
        for row in raw:
            if isinstance(row, dict) and row.get("instance_id"):
                panel[row["instance_id"]] = row
    elif isinstance(raw, dict):
        for key, value in raw.items():
            if isinstance(value, dict):
                panel[key] = value
print("traces:", len(traces), "| panel rows:", len(panel))

if not traces:
    raise SystemExit("No traces under /kaggle/working/results_*/traces. Run Cell 67 first.")


def steps_of(trace):
    for key in ("steps", "events", "trajectory", "turns", "records"):
        if isinstance(trace.get(key), list):
            return trace[key]
    return []


first = json.loads(traces[0].read_text(encoding="utf-8"))
print("top-level keys:", list(first)[:24])
sample = steps_of(first)
if sample:
    print("one step keys:", list(sample[0])[:24])
    print("first step preview:", json.dumps(sample[0])[:400])


def tool_name(step):
    for key in ("tool", "tool_name", "name", "function", "action"):
        value = step.get(key)
        if isinstance(value, str) and value:
            return value
        if isinstance(value, dict):
            for inner in ("name", "tool_name"):
                if isinstance(value.get(inner), str):
                    return value[inner]
    for key in ("tool_call", "call"):
        value = step.get(key)
        if isinstance(value, dict):
            for inner in ("name", "tool_name", "function"):
                found = value.get(inner)
                if isinstance(found, str):
                    return found
    return "?"


def stamp(step):
    for key in ("timestamp", "time", "start_time", "ts", "created_at"):
        value = step.get(key)
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            try:
                return float(value)
            except ValueError:
                continue
    return None


print()
header = f"{'task':16} {'steps':>5} {'edit':>4} {'submit':>6} {'gnaw':>5}  tools"
print(header)
print("-" * len(header))
summary = {"no_edit": [], "edited": [], "no_submit": [], "first_edit_after": []}
for path in traces:
    trace = json.loads(path.read_text(encoding="utf-8"))
    steps = steps_of(trace)
    names = [tool_name(step) for step in steps]
    counts = Counter(names)
    edits = [index for index, name in enumerate(names) if "edit" in name.lower()]
    submits = [index for index, name in enumerate(names) if "submit" in name.lower()]
    task = path.stem.replace("trace_", "")
    row = panel.get(task, {})
    tool_calls = row.get("tool_calls", len(steps))
    duration = row.get("duration", row.get("duration_s"))
    top = ", ".join(f"{name}:{count}" for name, count in counts.most_common(5))
    print(f"{task:16} {len(steps):>5} {len(edits):>4} {len(submits):>6} {str(tool_calls):>5}  {top}")
    if edits:
        summary["edited"].append(task)
        summary["first_edit_after"].append((task, edits[0] + 1))
    else:
        summary["no_edit"].append(task)
    if edits and not submits:
        summary["no_submit"].append(task)

print()
print("tasks with NO edit_file at all :", len(summary["no_edit"]), summary["no_edit"])
print("tasks that edited              :", len(summary["edited"]))
print("edited but never submitted     :", len(summary["no_submit"]), summary["no_submit"])
print("call number of the first edit  :", summary["first_edit_after"])
print()
print("If most dead tasks have zero edit_file, the locate stage is eating the clock and the fix is a hard")
print("edit-by-call-N rule plus cheaper thinking. If they edit but never submit, the fix is in verify.")

Cell 74

In [ ]:
# Cell 74 - fix Cell 73's schema guess. The traces are ATIF: steps have step_id/source/message/extra, so
# the tool names live inside "extra". This prints the real shape, reads the budget the harness actually
# enforced out of step 1, and then counts tools per task by scanning each step's JSON.
import json
import re
from collections import Counter
from pathlib import Path

ROOT = Path("/kaggle/working")
traces = sorted(ROOT.glob("results_*/traces/trace_*.json"))
TOOLS = ["run_command", "read_file", "edit_file", "write_file", "submit_patch", "get_status",
         "search_similar_code", "get_code_neighbors", "get_code_subgraph", "run_skill_script"]

print("traces:", len(traces))
first = json.loads(traces[0].read_text(encoding="utf-8"))
steps = first["steps"]
print("final_metrics:", json.dumps(first.get("final_metrics"))[:400])
print("sources seen:", Counter(step.get("source") for step in steps))
model_step = next((step for step in steps if step.get("source") != "user"), None)
if model_step:
    print("model step keys:", list(model_step))
    print("extra keys:", list(model_step.get("extra") or {}))
    print("extra preview:", json.dumps(model_step.get("extra"))[:600])
    print("message preview:", str(model_step.get("message"))[:300])

print()
print("=== the budget the harness actually enforced (from step 1) ===")
for path in traces[:4]:
    trace = json.loads(path.read_text(encoding="utf-8"))
    head = trace["steps"][0].get("message", "")
    block = head.split("## Task Budget", 1)[-1].split("##", 1)[0]
    print(path.stem.replace("trace_", ""), "|", " ".join(block.split())[:200])

print()
header = f"{'task':16} {'calls':>5} {'first_edit':>10} {'submit_at':>9}  tools"
print(header)
print("-" * len(header))
totals = Counter()
no_edit = []
for path in traces:
    trace = json.loads(path.read_text(encoding="utf-8"))
    counts = Counter()
    first_edit = None
    submit_at = None
    call_index = 0
    for step in trace["steps"]:
        if step.get("source") == "user":
            continue
        blob = json.dumps(step)
        hit = False
        for name in TOOLS:
            occurrences = blob.count(name)
            if occurrences:
                counts[name] += occurrences
                hit = True
                if name == "edit_file" and first_edit is None:
                    first_edit = call_index + 1
                if name == "submit_patch" and submit_at is None:
                    submit_at = call_index + 1
        if hit:
            call_index += 1
    totals.update(counts)
    task = path.stem.replace("trace_", "")
    top = ", ".join(f"{name}:{count}" for name, count in counts.most_common(4))
    print(f"{task:16} {call_index:>5} {str(first_edit):>10} {str(submit_at):>9}  {top}")
    if first_edit is None:
        no_edit.append(task)

print()
print("totals across all tasks:", dict(totals.most_common()))
print("tasks with no edit_file :", len(no_edit), no_edit)
print()
print("Read it like this: if run_command and read_file dominate and edit_file is absent or late, locate")
print("is eating the clock. If python3 -m pytest appears often, verify is.")
print("NOTE: substring counts include prompt text echoed in later steps, so treat them as an upper bound.")

Cell 75

In [ ]:
# Cell 75 - is the repo-scout skill actually working inside the sandbox? Every task made exactly two
# run_skill_script calls, which is the signature of a failure plus a retry. If the skill errors, the
# agent is grepping blind and the retrieval work never reaches the agent at all.
import json
from pathlib import Path

ROOT = Path("/kaggle/working")
traces = sorted(ROOT.glob("results_*/traces/trace_*.json"))
print("traces:", len(traces))

print()
print("=== every run_skill_script call, with its result ===")
for path in traces:
    trace = json.loads(path.read_text(encoding="utf-8"))
    task = path.stem.replace("trace_", "")
    for index, step in enumerate(trace["steps"], start=1):
        blob = json.dumps(step)
        if "run_skill_script" not in blob and "repo-scout" not in blob:
            continue
        calls = step.get("tool_calls")
        observation = step.get("observation")
        message = step.get("message") or ""
        print(f"--- {task} step {index} (source={step.get('source')})")
        if calls:
            print("    tool_calls:", json.dumps(calls)[:300])
        if message:
            print("    message   :", str(message)[:200].replace(chr(10), " "))
        if observation:
            text = json.dumps(observation) if not isinstance(observation, str) else observation
            print("    observation:", text[:600].replace(chr(10), " "))

print()
print("=== verdict per task: did the skill return candidates? ===")
for path in traces:
    trace = json.loads(path.read_text(encoding="utf-8"))
    blob = json.dumps(trace)
    looks_failed = any(token in blob for token in
                       ["No such file", "not found", "Traceback", "ModuleNotFoundError",
                        "SkillNotFound", "Permission denied", "returncode", "error"])
    has_candidates = '"candidates"' in blob or "candidate" in blob
    print(f"{path.stem.replace('trace_', ''):16} mentions_error={looks_failed!s:5} "
          f"mentions_candidates={has_candidates!s:5}")

Cell 76

In [ ]:
# Cell 76 - the decisive check: did run_skill_script return candidates, or fail again?
import json
from pathlib import Path

traces = sorted(Path("/kaggle/working").glob("results_*/traces/trace_*.json"))
print("traces:", len(traces))
seen = 0
for path in traces:
    trace = json.loads(path.read_text(encoding="utf-8"))
    task = path.stem.replace("trace_", "")
    for index, step in enumerate(trace["steps"], start=1):
        blob = json.dumps(step)
        if "run_skill_script" not in blob:
            continue
        observation = step.get("observation") or {}
        content = observation.get("content") if isinstance(observation, dict) else str(observation)
        try:
            payload = json.loads(content)
        except Exception:
            print(task, "step", index, "raw:", str(content)[:400])
            continue
        stdout = payload.get("stdout") or ""
        stderr = payload.get("stderr") or ""
        seen += 1
        print("===", task, "step", index, "| file:", payload.get("file_path"))
        print("    stdout chars:", len(stdout), "| contains candidates:", '"candidates"' in stdout)
        print("    stdout head :", stdout[:400].replace(chr(10), " "))
        print("    stderr head :", stderr[:400].replace(chr(10), " "))
print("skill calls seen:", seen)

Cell 77

In [ ]:
# Cell 77 - isolate the NEWEST trace and print its skill call in full. Cell 76 scanned every trace on
# disk, including the previous 20-task panel, and cut stderr at 400 chars just before the exception.
import json
import os
import time
from pathlib import Path

traces = sorted(Path("/kaggle/working").glob("results_*/traces/trace_*.json"),
                key=lambda p: p.stat().st_mtime, reverse=True)
print("traces on disk:", len(traces))
for path in traces[:6]:
    traced = json.loads(path.read_text(encoding="utf-8"))
    age_min = (time.time() - path.stat().st_mtime) / 60
    print(f"  {path.stem.replace('trace_', ''):16} age={age_min:7.1f} min  steps={len(traced['steps'])}")

newest = traces[0]
print()
print("=== newest trace:", newest.stem.replace("trace_", ""), "===")
trace = json.loads(newest.read_text(encoding="utf-8"))
found = False
for index, step in enumerate(trace["steps"], start=1):
    if "run_skill_script" not in json.dumps(step):
        continue
    found = True
    observation = step.get("observation") or {}
    content = observation.get("content") if isinstance(observation, dict) else str(observation)
    try:
        payload = json.loads(content)
    except Exception:
        print("raw observation:", str(content)[:2000])
        continue
    print("step", index, "| keys:", sorted(payload))
    if payload.get("error"):
        print("ERROR FIELD:", json.dumps(payload.get("error"))[:500])
    print("--- stdout (first 1200) ---")
    print((payload.get("stdout") or "")[:1200])
    print("--- stderr (full, up to 4000) ---")
    print((payload.get("stderr") or "")[:4000])
if not found:
    print("no run_skill_script call in the newest trace")